# Analysis notebook

This notebook contains cleaned analysis code for the manuscript on machine learning prediction of unsuppressed HIV viral load in Rwanda.

Patient-level CBS data are not included because they contain sensitive national HIV surveillance information. The code is provided for transparency and reproducibility of the analytic workflow.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import imblearn
import optuna
import lightgbm
import xgboost
import catboost
import scipy
import statsmodels

print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("Optuna:", optuna.__version__)
print("LightGBM:", lightgbm.__version__)
print("XGBoost:", xgboost.__version__)
print("CatBoost:", catboost.__version__)
print("SciPy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)

In [ ]:
# ------------------------------- #
# 📌 Importing Necessary Libraries #
# ------------------------------- #

import pandas as pd
import numpy as np
import os
import warnings
import sys

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings(action='ignore', category=UserWarning)

# ----------------------------- #
# 📌 Machine Learning Utilities #
# ----------------------------- #

from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     KFold, cross_val_score, RepeatedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import (f1_score, accuracy_score, recall_score, precision_score, balanced_accuracy_score,
                             confusion_matrix, roc_auc_score, mean_absolute_error,
                             mean_squared_error, r2_score)
from sklearn.decomposition import PCA
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures, RobustScaler)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import (LogisticRegression, LinearRegression, RidgeClassifier, SGDClassifier, Ridge, Lasso)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier,
                              GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor,
                              BaggingClassifier, BaggingRegressor, StackingClassifier, StackingRegressor,
                              ExtraTreesClassifier, ExtraTreesRegressor, VotingClassifier)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline

# --------------------------------- #
# 📌 Data Balancing (Imbalanced Data) #
# --------------------------------- #

try:
    from imblearn.ensemble import RUSBoostClassifier
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as imbpipeline
except ImportError:
    print("⚠️ Imbalanced-learn is not installed. Run: pip install imbalanced-learn")
    sys.exit(1)

# --------------------------------------------- #
# 📌 Hyperparameter Optimization & Feature Selection #
# --------------------------------------------- #

try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.integration import CatBoostPruningCallback
    from optuna.pruners import MedianPruner
except ImportError:
    print("⚠️ Optuna is not installed. Run: pip install optuna")

try:
    from hyperopt import hp, Trials
except ImportError:
    print("⚠️ Hyperopt is not installed. Run: pip install hyperopt")

# ------------------------------------- #
# 📌 Feature Selection with MLxtend #
# ------------------------------------- #

try:
    from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS
except ImportError:
    print("⚠️ MLxtend is not installed. Run: pip install mlxtend")

# ---------------------------------------- #
# 📌 Advanced Modeling (Boosting Algorithms) #
# ---------------------------------------- #

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
except ImportError:
    print("⚠️ LightGBM is not installed. Run: pip install lightgbm")

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
except ImportError:
    print("⚠️ CatBoost is not installed. Run: pip install catboost")

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError:
    print("⚠️ XGBoost is not installed. Run: pip install xgboost")

# ------------------------------------------------ #
# 📌 Additional Libraries for Model Interpretation #
# ------------------------------------------------ #

try:
    from shaphypetune import BoostSearch, BoostBoruta, BoostRFE, BoostRFA
except ImportError:
    print("⚠️ Shap-Hypetune is not installed. Run: pip install shap-hypetune")

# ------------------------------------- #
# 📌 Pandas Display Settings #
# ------------------------------------- #

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# --------------------------------------------- #
# 📌 Checking Versions (Ensure Compatibility) #
# --------------------------------------------- #

import sklearn
import imblearn

print(f"✅ scikit-learn version: {sklearn.__version__}")
print(f"✅ imbalanced-learn version: {imblearn.__version__}")


In [ ]:
# Read the first CSV file named CRF1.csv into a pandas DataFrame called df1
df1 = pd.read_csv('CRF1.csv')

# Read the second CSV file named CRF2.csv into a pandas DataFrame called df2
df2 = pd.read_csv('CRF2.csv')


# Define the common key column that exists in both df1 and df2
# This column will be used to match records between the two datasets
key = 'cbs_Index UPID'


# Keep only rows in df2 where the UPID exists in df1
# .isin(df1[key]) checks whether each UPID in df2 is also found in df1
# .copy() creates an independent copy of the filtered data to avoid warnings later
df2_filtered = df2[df2[key].isin(df1[key])].copy()


# Reset the row index of the filtered DataFrame
# drop=True prevents the old index from being added as a new column
# inplace=True updates df2_filtered directly
df2_filtered.reset_index(drop=True, inplace=True)


# Display the first 5 rows of the filtered DataFrame
df2_filtered.head()

In [ ]:
# Combine df1 and df2_filtered into one DataFrame by stacking rows
# ignore_index=True resets the row index after combining
df = pd.concat([df1, df2_filtered], axis=0, ignore_index=True)

# Display the first 5 rows of the combined DataFrame
df.head()

In [ ]:
# Filter the combined DataFrame
# Condition 1: Keep records with a non-missing cbs_Index UPID
# Condition 2: Remove records from the "Training" organisation unit
df = df[
    df['cbs_Index UPID'].notnull() &
    (df['Organisation unit name'] != 'Training')
].copy()

In [ ]:
# Loop through all columns in df that have object/string data type
for col in df.select_dtypes(include=['object']).columns:
    
    # Convert all text values in the selected column to lowercase
    # This helps standardize text values, for example "Male", "MALE", and "male" become "male"
    df[col] = df[col].str.lower()

# Display a random sample of 10 rows from the DataFrame
df.sample(10)

In [ ]:
# Read the Excel file named "cbs_Facility .xlsx" into a pandas DataFrame called df30
# This file contains facility-related information like province, district,....
df30 = pd.read_excel('cbs_Facility .xlsx')

# Display the first 5 rows of the DataFrame to inspect the data
df30.head()

In [ ]:
# Loop through all columns in the facility DataFrame
for col in df30.columns:
    
    # Check whether the column contains text/string values
    if df30[col].dtype == 'object':
        
        # Convert all text values in that column to lowercase
        # This helps ensure facility names match correctly during merging
        df30[col] = df30[col].str.lower()
        

# Merge the main DataFrame df with the facility DataFrame df30
# left_on='Organisation unit name' means the merge key from df is Organisation unit name
# right_on='Facility_name' means the merge key from df30 is Facility_name
# how='left' keeps all rows from df, even if no matching facility is found in df30
# indicator=True creates a new column called _merge showing whether each row matched
df = pd.merge(
    df,
    df30,
    left_on='Organisation unit name',
    right_on='Facility_name',
    how='left',
    indicator=True
)

In [ ]:
# Checking for missing values
round((df.isnull().sum()/df.isnull().count())*100,2)

In [ ]:
df.shape

In [ ]:
# Filter the DataFrame to include only rows where 'cbs_Viral Load' is not missing and greater than 0
df = df[df['cbs_Viral Load'].notnull() & (df['cbs_Viral Load'] > 0)]

# Filter the DataFrame to keep only rows where 'cbs_Index UPID' is not missing and 'Organisation unit name' is not 'Training'
#df = df[df['cbs_Index UPID'].notnull() & (df['Organisation unit name'] != 'Training')]

# Filter the DataFrame to keep only rows where 'cbs_WHO Stage' is not missing
df = df[df['cbs_WHO Stage'].notnull()]

# Replace 'unknown' values in the 'cbs_pregnant' column with NaN
df['cbs_pregnant'] = df['cbs_pregnant'].replace('unknown', np.nan)

# Identify cbs_Index UPID with 'death' or 'stopped' outcome
dead_indices = df[df['cbs_Client outcome'].isin(['death', 'stopped'])]['cbs_Index UPID']

# Replace 'transfer out (to)' with 'alive and on art'
df['cbs_Client outcome'] = df['cbs_Client outcome'].replace('transfer out (to)', 'alive and on art')

# Remove rows where 'cbs_Index UPID' is in the dead_indices list
df = df[~df['cbs_Index UPID'].isin(dead_indices)]


# Replace 'transfer out (to)' with 'alive and on art'
df['cbs_Client outcome'] = df['cbs_Client outcome'].replace('transfer out (to)', 'alive and on art')

# Remove rows where 'cbs_Index UPID' is in the dead_indices list
df = df[~df['cbs_Index UPID'].isin(dead_indices)]

# Check the new shape of the dataset
df.shape


In [ ]:
# Count how many records fall under each category of client outcome
# This helps you understand the distribution of outcomes, for example:
# active, transferred out, died, lost to follow-up, stopped treatment, etc.
df['cbs_Client outcome'].value_counts()

In [ ]:
# Count viral load tests per patient
patients_with_one_test = df['cbs_Index UPID'].value_counts()

# Get only those with one test
single_test_patients = patients_with_one_test[patients_with_one_test == 1].index
print(single_test_patients.shape)

# Remove single-test patients
df = df[~df['cbs_Index UPID'].isin(single_test_patients)]

# Print new shape to confirm the change
print(f"✅ New dataset shape after removing single-test patients: {df.shape}")

In [ ]:
# Creating a function to convert columns into datetime dtype
def To_Datetime(data, column):
    data[column] = pd.to_datetime(data[column], errors='coerce')
    return data[column]

In [ ]:
# convert the columns we want into datetime
To_Datetime(df,'cbs_Viral load collection date')
To_Datetime(df,'cbs_CD4 collection date')
To_Datetime(df,'cbs_date of birth')
To_Datetime(df,'cbs_Last pharmacy pick up date')
To_Datetime(df,'cbs_OI Diagnosis date')
To_Datetime(df,'Enrollment date in CBS')

In [ ]:
%%time

def preprocess_data(df):
    # Convert 'cbs_Viral load collection date' to datetime with error handling
    df['cbs_Viral load collection date'] = pd.to_datetime(
        df['cbs_Viral load collection date'], errors='coerce'
    )
    
    # Drop rows with invalid dates (NaT values)
    df = df.dropna(subset=['cbs_Viral load collection date'])
    
    # Filter out dates outside the range 2000-01-01 to 2025-01-01
    mask = (
        (df['cbs_Viral load collection date'] >= '2000-01-01'))
    
    df = df.loc[mask]

    # Sort by 'cbs_Index UPID' and 'cbs_Viral load collection date'
    df = df.sort_values(by=['cbs_Index UPID', 'cbs_Viral load collection date'])

    # Drop duplicates, keeping the last occurrence within each year for each 'cbs_Index UPID'
    df['year'] = df['cbs_Viral load collection date'].dt.year
    df = df.drop_duplicates(subset=['cbs_Index UPID', 'year'], keep='last')
    df = df.drop(columns='year')

    return df

# Apply the function
df = preprocess_data(df)

df.shape


In [ ]:
# Checking for missing values
round((df.isnull().sum()/df.isnull().count())*100,2)

In [ ]:
def drop_high_na_columns(df):
    # Determine columns to drop based on threshold
    threshold = 0.96 * len(df)
    cols_to_drop = df.columns[df.isna().sum() > threshold]

    # Drop the columns
    df.drop(cols_to_drop, axis=1, inplace=True)

    # Return percentage of missing values for each column
    return (df.isnull().sum() / df.isnull().count()) * 100

# Apply the function to each DataFrame and print the percentage of missing values
print(drop_high_na_columns(df))
print("-" * 100)


In [ ]:
# Replace spaces with underscores and remove hyphens in column names
df.columns = [col.replace(" ", "_").replace("-", "") for col in df.columns]

In [ ]:
df.head()

In [ ]:
# Convert the cbs_stable column to numeric values
# errors='coerce' changes any non-numeric values into missing values (NaN)
# astype('Int64') converts the column to pandas nullable integer type
# This allows the column to contain integers while still keeping missing values as <NA>
df['cbs_stable'] = pd.to_numeric(df['cbs_stable'], errors='coerce').astype('Int64')

In [ ]:
# ✅ Replace 'nan' string values with "None" in categorical features
df['cbs_Client_TPT_outcome'] = df['cbs_Client_TPT_outcome'].replace('nan', "None")


# ✅ Convert "nan" and empty strings to np.nan
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df.replace("nan", np.nan, inplace=True)

df['cbs_Client_TPT_outcome'] = df['cbs_Client_TPT_outcome'].replace(np.nan, "None")

In [ ]:
# Check the size of the DataFrame
df.shape

In [ ]:
# Get a list of all column names in the DataFrame
df.columns.tolist()

In [ ]:
# Count the number of records in each adherence category
df['cbs_Client_adherence'].value_counts(dropna=False)

In [ ]:
# Recode values in the pregnancy column from French or misspelled values into standardized English categories
df['cbs_pregnant'] = df['cbs_pregnant'].replace({
    'non': 'no',              # French for "no"
    'mo': 'no',               # Correct possible typo/misspelling to "no"
    'oui': 'yes',             # French for "yes"
    'non connu': 'unknown'    # French for "not known"
})


# Recode values in the marital status column from French into standardized English categories
df['cbs_marital_status'] = df['cbs_marital_status'].replace({
    'marie': 'married',                 # French for "married"
    'cohabite': 'co-habiting',          # French for "cohabiting"
    'sans abri': 'homeless',            # French for "homeless"
    'celibataire': 'single',            # French for "single"
    'veuf': 'widow',                    # French for "widow/widower"
    'divorce ou separe': 'divorced'     # French for "divorced or separated"
})


# Recode values in the employment status column from French into standardized English categories
df['cbs_employment_status'] = df['cbs_employment_status'].replace({
    'employe': 'employed',        # French for "employed"
    'sans emploi': 'unemployed'   # French for "unemployed"
})

In [ ]:
# Mapping known synonyms to a standard term
mapping = {
    'n': 'none',
    'non infection':'none',
    'pas dio':'none',
    'no oi': 'none',
    'no': 'none',
    'non': 'none',
    'non oi': 'none',
    'noone': 'none',
    'no one': 'none',
    'negatif': 'none',
    'neg': 'negative',
    'herpes zooster': 'herpes zoster',
    'herpes zostes': 'herpes zoster',
    'zoona': 'zona',
    'zooster': 'zoster',
    'tbc': 'tuberculosis',
    'tb': 'tuberculosis',
    'tuberculose':'tuberculosis',
    'non probleme':'none',
    'ug': 'unknown',
    'tht': 'unknown',
    'i': 'unknown',
    'rien':'none',
    'not':'none',
    'pas':'none',
    'o':'none',
    'iha':'unknown',
    'oui':'unknown',
    'n':'none',
    'uti':'unknown',
    'ivrs':'unknown'
}



# Mapping known synonyms to a standard term
df['cbs_OI_Name'] = df['cbs_OI_Name'].replace(mapping)
# Removing special characters and extra spaces
df['cbs_OI_Name'] = df['cbs_OI_Name'].str.replace('[^a-z\s]', '', regex=True).str.strip()


In [ ]:
# Step 1: Identify French terms and their English equivalents

french_to_english = {
    'tuberculose': 'tuberculosis',
    'siphilis': 'syphilis',
    'infections des voies respiratoires superieurs': 'upper respiratory tract infection',
    'zona': 'shingles',
    'fievre prolongee': 'prolonged fever',
    'diarrhee': 'diarrhea',
    'candidose orale': 'oral candidiasis',
    'candidose bucale': 'oral candidiasis',
    'diarrhea chronique': 'chronic diarrhea',
    'tbc pulmpnaire': 'pulmonary tuberculosis',
    'stomatite ulcerative': 'ulcerative stomatitis',
    'perte de poids': 'weight loss',
    'perte depoids': 'weight loss',
    'pneumonie': 'pneumonia',
    'eruption cutane': 'skin rash',
    'erruption cutannee': 'skin rash',
    'vaginal dischange': 'vaginal discharge',
    'ist': 'STI',
    'malnutrition severe': 'severe malnutrition',
    'tuberculose pulmonaire': 'pulmonary tuberculosis',
    'herpes zosters': 'herpes zoster',
    'sphilis': 'syphilis',
    'candidose': 'candidiasis',
    'infection vih asymptomatique': 'asymptomatic HIV infection',
    'pulmonary tb': 'pulmonary tuberculosis',
    'weigh loss': 'weight loss',
    'syphilis': 'syphilis',
    'dermatoses': 'dermatosis',
    'ecoulement vaginal': 'vaginal discharge',
    'meningite a cryptocoque': 'cryptococcal meningitis',
    'pas dio': 'none',
    'manifestation muco cutan\xc3\xa9e mineur': 'minor mucocutaneous manifestation',
    'negative': 'none',
    'pulmonary tb mucocutaneu': 'pulmonary tuberculosis',
    'diarhea': 'diarrhea',
    'asymptomatique': 'Asymptomatic',
    'n': 'none',
    'manifestation mucocutanes mineurs': 'minor mucocutaneous manifestations',
    'diarrhe': 'diarrhea',
    'mucoses cutannees': 'mucocutaneous',
    'manifestation cutanee': 'cutaneous manifestation',
    'manufestation muco cutan\xc3\xa9e': 'mucocutaneous manifestation',
    'tauxdiarrhee': 'diarrhea',
    'diarrhee chronique': 'chronic diarrhea',
    'dns': 'unknown',
    'skin rash': 'skin rashes',
    'chronic diarrhoea': 'chronic diarrhea',
    'candidose orale diarrh\xc3\xa9e chronique': 'chronic diarrhea',
    'no oi diagnosed': 'none',
    'diarrhee chronique tb': 'chronic diarrhea',
    'erruption cutane': 'skin rashes',
    'no sti': 'none',
    'cyyptococal menengite': 'cryptococcal meningitis',
    'severe malnutrition': 'severe malnutrition',
    'sarcome de kaposi candidose orale m\xc3\xa9ningite cryptocoque': 'cryptococcal meningitis',
    'diarrheafeverinoxphe prolongeanemiemalnutrition': 'diarrhea',
    'infection des vois respsup': 'upper respiratory tract infection',
    'mucocutanee minereres': 'minor mucocutaneous',
    'aucune': 'none',
    'majocoses cutaneee': 'cutaneous mycoses',
    'manifestations mucocutan\xc3\xa9es mineures': 'minor mucocutaneous manifestations',
    'tb pulmonaire': 'pulmonary tuberculosis',
    'non infectiion': 'none',
    'manifastation cutane': 'cutaneous manifestation',
    'manifestation muco cutan\xc3\xa9e': 'mucocutaneous manifestation',
    'manifestation muco cutanee mineure': 'minor mucocutaneous manifestations',
    'asympto': 'asymptomatic',
    'eruption cutanes': 'skin rashes',
    'perte depoidsdiarrhea sangrate': 'weight loss',
    'affection cutane': 'skin conditions',
    'affectino muco cutanees  mineures': 'minor mucocutaneous manifestations',
    'manifestation mucocutanee': 'mucocutaneous manifestation',
    'eruption cutanee': 'skin rashes',
    'perte deprids  pourcents': 'weight loss',
    'infection vih symptomatique': 'symptomatic HIV infection',
    'pas infecions opportinistes': 'none',
    'infection bacterienne gravecandidose oralecachexie': 'serious bacterial infection',
    'candidose tb psitive': 'candidiasis with positive tuberculosis',
    'diarrhe chronique inexpliquemois': 'chronic diarrhea',
    'manifestation cutanees mineurs': 'minor cutaneous manifestations',
    'manifestations cutanes mineurs': 'minor cutaneous manifestations',
    '': 'unknown',
    'no infection': 'none',
    'manufestation muco cutanee': 'mucocutaneous manifestation',
    'manifestation muco cutanee': 'mucocutaneous manifestation',
    'manifestation muco cutanee mineur': 'minor mucocutaneous manifestations',
    'pulmonary tb mucocutaneur': 'pulmonary tuberculosis',
    'confirmed tuberculosis': 'tuberculosis',
    'yes': 'unknown',
    'candidose orale diarrhee chronique': 'chronic diarrhea',
    'manifestations mucocutanees mineures': 'minor mucocutaneous manifestations',
    'any': 'unknown',
    'ras': 'unknown',
    'syphillis': 'syphilis',
    'candidose oropharingee': 'oropharyngeal candidiasis',
    'Asymptomatic': 'asymptomatic',
    'chronic diarrhea weight loss': 'chronic diarrhea',
    'minor mucocutaneous conditions': 'minor mucocutaneous infection',
}

# Apply the mapping to the OI_Name column
df['cbs_OI_Name'] = df['cbs_OI_Name'].map(french_to_english)


In [ ]:
# Mapping from 'cbs_Initial ART Regimen' to 'Initial ART regimen'
regimen_mapping = {
    'tdf+3tc+dtg': 'TDF/3TC/DTG',
    'tenofovir/lamivudine/efavirenz(300/300/600mg)': 'TDF/3TC+EFV',
    'tenofovir/lamivudine(300/300mg)+nevirapine200mg': 'TDF/3TC+NVP',
    'zidovudine/lamivudine/nevirapine(300/150/200mg)': 'AZT/3TC+NVP',
    'tdf+3tc+efv': 'TDF/3TC+EFV',
    'autre': 'UNKNOWN',
    'abacavir/lamivudine(600/300mg)+efavirenz600mg': 'ABC/3TC+EFV',
    'abacavir/lamivudine(600/300mg)+nevirapine200mg ': 'ABC/3TC+NVP',
    'azt+3tc+efv': 'AZT/3TC+EFV',
    'tenofovir/lamivudine/dolutegravir(300/300/50mg)':'TDF/3TC/DTG',
    'abacavir/lamivudine(600/300)+dolutegravir50mg':'ABC/3TC+DTG',
    'azt+3tc+nvp':'AZT/3TC+NVP',
    'tdf+ftc+nvp':'TDF/FTC+NVP',
    'd4t+3tc+nvp':'D4T+3TC+NVP',
    'zidovudine/lamivudine(300/150mg)+efavirenz600mg':'AZT/3TC+EFV',
    'tenofovir/lamivudine(300/300mg)+atazanavir-r(300/1':'TDF/3TC+ATV/r',
    'zidovudine/lamivudine(300/150mg)+atazanavir-r(300-':'AZT/3TC+ATV/r',
    'azt+3tc+dtg':'AZT/3TC+DTG',
    'abacavir/lamivudine(600/300mg)+atazanavir-r(300/10':'ABC/3TC+ATV/r',
    'tdf300mg+zidovudine/lamivudine(300/150mg)+lopinavi':'TDF/3TC+LPV/r',
    'tdf+3tc+kaletra':'TDF/3TC+LPV/r',
    'd4t+3tc+efv':'D4T+3TC+EFV',
    'zidovudine/lamivudine(300/150mg)+lopinavir-r(200/5':'AZT/3TC+LPV/r',
    'raltegravir400mg+darunavir600mg+ritonavir100mg+ten':'RAL+DRV/r',
    'tdf+3tc+lpv/r':'TDF/3TC+LPV/r',
    'raltegravir400mg+etravirine100mg+darunavir600mg+ri':'RAL+ETV+DRV/r',
    'azt+3tc+lpv/r':'AZT/3TC+LPV/r',
    'tenofovir/lamivudine(300/300mg)+lopinavir-r(200/50':'TDF/3TC+LPV/r',
    'abacavir/lamivudine(600/300mg)+lopinavir-r(200/50m':'ABC/3TC+LPV/r',
    'tdf300mg+zidovudine/lamivudine(300/150mg)+atazanav':'TDF+AZT/3TC+ATV/r'

}

# Apply the mapping to create a new column
df['cbs_Current_ART_Regimen'] = df['cbs_Current_ART_Regimen'].map(regimen_mapping)


In [ ]:
# Mapping from 'cbs_Initial ART Regimen' to 'Initial ART regimen'
regimen_mapping = {
    'tdf+3tc+dtg': 'TDF/3TC/DTG',
    'tenofovir/lamivudine/efavirenz(300/300/600mg)': 'TDF/3TC+EFV',
    'tenofovir/lamivudine(300/300mg)+nevirapine200mg': 'TDF/3TC+NVP',
    'zidovudine/lamivudine/nevirapine(300/150/200mg)': 'AZT/3TC+NVP',
    'tdf+3tc+efv': 'TDF/3TC+EFV',
    'autre': 'UNKNOWN',
    'abacavir/lamivudine(600/300mg)+efavirenz600mg': 'ABC/3TC+EFV',
    'abacavir/lamivudine(600/300mg)+nevirapine200mg ': 'ABC/3TC+NVP',
    'azt+3tc+efv': 'AZT/3TC+EFV',
    'tenofovir/lamivudine/dolutegravir(300/300/50mg)':'TDF/3TC/DTG',
    'abacavir/lamivudine(600/300)+dolutegravir50mg':'ABC/3TC+DTG',
    'azt+3tc+nvp':'AZT/3TC+NVP',
    'tdf+ftc+nvp':'TDF/FTC+NVP',
    'd4t+3tc+nvp':'D4T+3TC+NVP',
    'zidovudine/lamivudine(300/150mg)+efavirenz600mg':'AZT/3TC+EFV',
    'tenofovir/lamivudine(300/300mg)+atazanavir-r(300/1':'TDF/3TC+ATV/r',
    'zidovudine/lamivudine(300/150mg)+atazanavir-r(300-':'AZT/3TC+ATV/r',
    'azt+3tc+dtg':'AZT/3TC+DTG',
    'abacavir/lamivudine(600/300mg)+atazanavir-r(300/10':'ABC/3TC+ATV/r',
    'tdf300mg+zidovudine/lamivudine(300/150mg)+lopinavi':'TDF/3TC+LPV/r',
    'tdf+3tc+kaletra':'TDF/3TC+LPV/r',
    'd4t+3tc+efv':'D4T+3TC+EFV',
    'zidovudine/lamivudine(300/150mg)+lopinavir-r(200/5':'AZT/3TC+LPV/r',
    'raltegravir400mg+darunavir600mg+ritonavir100mg+ten':'RAL+DRV/r',
    'tdf+3tc+lpv/r':'TDF/3TC+LPV/r',
    'raltegravir400mg+etravirine100mg+darunavir600mg+ri':'RAL+ETV+DRV/r',
    'azt+3tc+lpv/r':'AZT/3TC+LPV/r',
    'tenofovir/lamivudine(300/300mg)+lopinavir-r(200/50':'TDF/3TC+LPV/r',
    'abacavir/lamivudine(600/300mg)+lopinavir-r(200/50m':'ABC/3TC+LPV/r',
    'tdf300mg+zidovudine/lamivudine(300/150mg)+atazanav':'TDF+AZT/3TC+ATV/r'

}

# Apply the mapping to create a new column
df['cbs_Initial_ART_regimen'] = df['cbs_Initial_ART_regimen'].map(regimen_mapping)


In [ ]:
# Checking for missing values
round((df.isnull().sum()/df.isnull().count())*100,2)

In [ ]:
# corrected list (fixed comma after 'month', removed dup 'cbs_District')
want_drop = [
    'Initial_ART_regimen=_Current_ART_regimen', 'cbs_CD4_collection_date', 'cbs_CD4_result_return_date',
    'cbs_Cell', 'cbs_Counselled_on_Linkage', 'cbs_Date_Linked', 'cbs_If_yes_TPT_Start_Date',
    'cbs_Initiated_on_TPT', 'cbs_Last_pharmacy_pick_up_date', 'cbs_Linked_to_Treatment',
    'cbs_Linked_to_Treatment_at_this_facility', 'cbs_Reporting_staff_Name', 'cbs_Reporting_staff_email',
    'cbs_Reporting_staff_phone', 'cbs_Sector', 'cbs_Testing_clinic', 'cbs_Testing_facility_name',
    'cbs_Village', 'cbs_Viral_load_result_return_date', 'cbs_WHO_Stage_Date', 'Tracnet_No',
    'Event_status', 'Program_status', 'Created_by', 'Last_updated_by', 'Enrollment_date_in_CBS',
    'Last_updated_on', 'cbs_Outcome_Date', 'cbs_Client_visit_date', 'cbs_If_different_ART_change_date',
    'cbs_Client_TPT_Outcome_Date', 'SubDistrict', 'District', 'Sector', 'Facility_name', 'cbs_District',
    'cbs_OI_Diagnosis_date', 'Organisation_unit_name', 'Last_Viral_Load', 'Last_Viral_Load_Test_Date',
    'cbs_Any_Changes_in_Demographic_Information', 'cbs_date_of_birth', 'Report_date', 'cbs_initiated_on_tpt',
    'cbs_Date_of_HIV_positive_result_confirmation', 'cbs_If_currently_on_ART','adherence_sd_3visits',
    'cbs_Any_Changes_in_or_Undisclosed_Sexual_Partners', 'cbs_Sex_with_multiple_partners',
    'cbs_Sex_with_commercial_sex_worker', 'cbs_Any_vaginal_or_anal_sex_without_condoms_in_last_12_months',
    'cbs_Date_de_debut_des_ARVs', 'cbs_Change_in_regimen', 'cbs_Index_UPID','delta_cd4_recent',
    'cbs_Number_of_sexual_partners_in_last_3_months', 'cbs_Number_of_lifetime_sexual_partners',
    'cbs_Viral_load_collection_date', 'cbs_Viral_Load', 'cbs_Client_adherence', 'month',
    'cbs_pregnant', 'num_alive_children_under_15', 'cbs_reporting_staff_name', 'facility_name',
    'cbs_date_linked', 'cbs_who_stage_date', 'report_date','peer_suppression_diff','cd4_slope',
    'initial_art_regimen_current_art_regimen','cbs_cd4_collection_date','cbs_cd4_result_return_date',
    'cbs_client_visit_date','cbs_date_linked',
    'cbs_if_currently_on_art_where_initiated_art','cbs_if_yes_tpt_start_date','cbs_last_pharmacy_pick_up_date',
    'cbs_oi_diagnosis_date','cbs_recency_assay_test_done',
    'cbs_recency_assay_test_results','cbs_reporting_staff_name',
    'cbs_testing_clinic','cbs_testing_facility_name','cbs_viral_load_collection_date',
    'cbs_viral_load_result_return_date','cbs_who_stage_date','tracnet_no','cbs_index_upid','cbs_date_of_birth',
    'created_by','last_updated_by','report_date','enrollment_date_in_cbs',
    'last_updated_on','cbs_client_tpt_outcome_date','cbs_if_yes_completed_3_months','cbs_outcome_date','district',
    'sector','facility_name','merge','organisation_unit_name','cbs_date_de_debut_des_arvs','subdistrict',
    'prev_visit_date','cbs_date_of_hiv_positive_result_confirmation','month','cbs_Change_in_regimen',
    'last_outcome_date','cbs_Viral_Load','Late_ART_Initiation','cbs_OI_Name','cbs_stable_category','prev_outcome',
    'cbs_WHO_Stage','Num_Suppressed','missed_visit_rate','regimen_change_flag','Suppression_Rate','A_to_L',
    'Age_at_ART_start','cbs_Any_Changes_in_or_Undisclosed_Sexual_Partners','cbs_Sex_with_multiple_partners',
    'cbs_Sex_with_commercial_sex_worker','cbs_Any_vaginal_or_anal_sex_without_condoms_in_last_12_months',
]

# robust, case-insensitive mapping to actual df columns
lower_to_actual = {c.lower(): c for c in df.columns}
want_drop_unique = sorted(set(want_drop))  # dedupe

to_drop_present = [lower_to_actual[name.lower()]
                   for name in want_drop_unique
                   if name.lower() in lower_to_actual]

missing = [name for name in want_drop_unique
           if name.lower() not in lower_to_actual]

print("Dropping (present):", to_drop_present)
if missing:
    print("Skipping (not in df):", missing)

df.drop(columns=to_drop_present, inplace=True)
df.shape


In [ ]:
# Display a concise summary of the DataFrame
df.info()

In [ ]:
%%time
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

pd.options.mode.use_inf_as_na = True

# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────
def safe_reset_index(obj: pd.DataFrame) -> pd.DataFrame:
    idx_name = getattr(obj.index, "name", None)
    if idx_name and idx_name in obj.columns:
        return obj.reset_index(drop=True)
    return obj.reset_index()

def dt_days(lhs, rhs):
    """
    Robust difference in days between datetimes.
    Works for scalar Timestamp vs Timestamp, Series vs Series, or Series vs scalar.
    Returns:
      - Series when any input is a Series/Index
      - float when both inputs are scalars
    """
    lhs = pd.to_datetime(lhs)
    rhs = pd.to_datetime(rhs)

    lhs_is_arr = isinstance(lhs, (pd.Series, pd.Index))
    rhs_is_arr = isinstance(rhs, (pd.Series, pd.Index))

    if lhs_is_arr or rhs_is_arr:
        if not lhs_is_arr:
            lhs = pd.Series(lhs, index=getattr(rhs, "index", None))
        if not rhs_is_arr:
            rhs = pd.Series(rhs, index=getattr(lhs, "index", None))
        return (lhs - rhs).dt.days

    td = lhs - rhs
    if pd.isna(td):
        return np.nan
    try:
        return td / np.timedelta64(1, "D")
    except Exception:
        return td.days + td.seconds / 86400.0

# ──────────────────────────────────────────────────────────────────────────────
# 0) Global cleanup + case-insensitive column resolver
# ──────────────────────────────────────────────────────────────────────────────
df.replace(['unknown', 'missing', 'NA', 'n/a', ''], np.nan, inplace=True)

# Case-insensitive resolver for any column name
lower_to_actual = {c.lower(): c for c in df.columns}
def C(name: str) -> str:
    """Return actual df column name for a case-insensitive query."""
    return lower_to_actual.get(name.lower(), name)

# Coerce *any* date-like columns (case-insensitive)
canonical_date_cols = [
    'cbs_date_de_debut_des_arvs','cbs_date_of_hiv_positive_result_confirmation',
    'cbs_date_of_birth','report_date','cbs_client_visit_date',
    'cbs_viral_load_collection_date','cbs_viral_load_result_return_date',
    'cbs_if_yes_tpt_start_date','cbs_cd4_collection_date',
    'cbs_cd4_result_return_date','cbs_last_pharmacy_pick_up_date',
    'cbs_date_linked','enrollment_date_in_cbs','cbs_who_stage_date'
]
present_from_canonical = [C(c) for c in canonical_date_cols if c in lower_to_actual]
auto_dates = [c for c in df.columns if any(k in c.lower() for k in ['date', '_dt', 'datetime'])]
date_cols_to_parse = sorted(set(present_from_canonical + auto_dates))
if date_cols_to_parse:
    df[date_cols_to_parse] = df[date_cols_to_parse].apply(pd.to_datetime, errors='coerce')
print("Parsed date columns:", date_cols_to_parse)

# ──────────────────────────────────────────────────────────────────────────────
# 1) Basic sanitization
# ──────────────────────────────────────────────────────────────────────────────
if C('cbs_stable') in df.columns and C('cbs_stable_category') in df.columns:
    df[C('cbs_stable_category')] = np.where(
        df[C('cbs_stable')].isna(), np.nan, df[C('cbs_stable_category')]
    )
    df.loc[df[C('cbs_stable')] == 0, C('cbs_stable_category')] = 'no_category'

if C('cbs_Number_of_lifetime_sexual_partners') in df.columns:
    df.loc[df[C('cbs_Number_of_lifetime_sexual_partners')] > 100,
           C('cbs_Number_of_lifetime_sexual_partners')] = np.nan

condom_col = C('cbs_Any_vaginal_or_anal_sex_without_condoms_in_last_12_months')
if condom_col in df.columns:
    df[condom_col] = (
        df[condom_col].astype(str).str.strip().str.lower()
          .replace({'nan': np.nan, 'refuse': np.nan, 'unknown': np.nan})
          .astype('category')
    )
    if 0 not in df[condom_col].cat.categories:
        df[condom_col] = df[condom_col].cat.add_categories([0])

# Demographics
if {C('cbs_Date_de_debut_des_ARVs'), C('cbs_date_of_birth')}.issubset(df.columns):
    df['Age_at_ART_start'] = (
        df[C('cbs_Date_de_debut_des_ARVs')].dt.year - df[C('cbs_date_of_birth')].dt.year
    )

# Remove invalid records
if {C('cbs_Viral_load_collection_date'), C('cbs_Date_de_debut_des_ARVs')}.issubset(df.columns):
    df = df[df[C('cbs_Viral_load_collection_date')] >= df[C('cbs_Date_de_debut_des_ARVs')]]

if {C('cbs_date_of_birth'), C('cbs_Date_de_debut_des_ARVs'),
    C('cbs_Date_of_HIV_positive_result_confirmation'),
    C('Report_date'), C('cbs_Viral_load_collection_date')}.issubset(df.columns):
    df = df[
        ~((df[C('cbs_date_of_birth')] > df[C('cbs_Date_de_debut_des_ARVs')]) |
          (df[C('cbs_Date_de_debut_des_ARVs')] < df[C('cbs_Date_of_HIV_positive_result_confirmation')]) |
          (df[C('Report_date')] < df[C('cbs_Viral_load_collection_date')]))
    ]

# Sort by client and date
if {C('cbs_Index_UPID'), C('cbs_Viral_load_collection_date')}.issubset(df.columns):
    df = df.sort_values(by=[C('cbs_Index_UPID'), C('cbs_Viral_load_collection_date')])

# ART features that are valid at last row (the test row) — OK as features
if {C('cbs_Viral_load_collection_date'), C('cbs_Date_de_debut_des_ARVs')}.issubset(df.columns):
    df['Total_Time_on_ART'] = dt_days(
        df[C('cbs_Viral_load_collection_date')], df[C('cbs_Date_de_debut_des_ARVs')]
    )
if {C('cbs_Date_de_debut_des_ARVs'), C('cbs_Date_of_HIV_positive_result_confirmation')}.issubset(df.columns):
    df['Days_HIV_Diagnosis_to_ART'] = dt_days(
        df[C('cbs_Date_de_debut_des_ARVs')], df[C('cbs_Date_of_HIV_positive_result_confirmation')]
    )

# ──────────────────────────────────────────────────────────────────────────────
# 2) Build: (a) last-test view (label row) and (b) history view excluding it
# ──────────────────────────────────────────────────────────────────────────────
df_last_test = (
    df.sort_values(by=[C('cbs_Index_UPID'), C('cbs_Viral_load_collection_date')])
      .drop_duplicates(subset=C('cbs_Index_UPID'), keep='last')
)[[C('cbs_Index_UPID'), C('cbs_Viral_Load'), C('cbs_Viral_load_collection_date')]].copy()

df_last_test['Last_Viral_Load'] = df_last_test[C('cbs_Viral_Load')]
df_last_test.rename(columns={C('cbs_Viral_load_collection_date'): 'Last_Viral_Load_Test_Date'}, inplace=True)
df_last_test['Last_Viral_Load_Test_Date'] = pd.to_datetime(
    df_last_test['Last_Viral_Load_Test_Date'], errors='coerce'
)
# Label for modeling later (do NOT include as feature)
df_last_test['Viral_Load'] = (df_last_test['Last_Viral_Load'] >= 200).astype(int)

# history = everything strictly before the last test row per client
df_no_last_test = (
    df.groupby(C('cbs_Index_UPID'), group_keys=False)
      .apply(lambda x: x.iloc[:-1])
      .reset_index(drop=True)
)

# ──────────────────────────────────────────────────────────────────────────────
# 3) Features from HISTORY ONLY (no leakage)
# ──────────────────────────────────────────────────────────────────────────────

# ---- Current age (at last *historical* report) ----
if {C('Report_date'), C('cbs_date_of_birth'), C('cbs_Index_UPID')}.issubset(df.columns):
    age_days = dt_days(df[C('Report_date')], df[C('cbs_date_of_birth')])
    age_years_last_hist = (age_days // 365).astype(float).groupby(df[C('cbs_Index_UPID')]).last()
    df_last_test['current_age'] = df_last_test[C('cbs_Index_UPID')].map(age_years_last_hist.to_dict())

# ---- Viral-load counts & suppression rate ----
if C('cbs_Viral_Load') in df_no_last_test.columns:
    agg_vl = df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Viral_Load')].agg(
        Num_VL_Tests='count',
        Num_Suppressed=lambda s: (pd.to_numeric(s, errors='coerce') < 200).sum()
    )
    agg_vl['Suppression_Rate'] = (
        agg_vl['Num_Suppressed'] / agg_vl['Num_VL_Tests']
    ).replace([np.inf, -np.inf], np.nan)
    df_last_test = df_last_test.merge(agg_vl, left_on=C('cbs_Index_UPID'), right_index=True, how='left')

# ---- CD4 dynamics ----
def _safe_cd4_slope(g):
    g = g.dropna(subset=[C('cbs_cd4_collection_date'), C('cbs_cd4_count_results')])
    if len(g) < 2: 
        return np.nan
    X = dt_days(g[C('cbs_cd4_collection_date')], g[C('cbs_cd4_collection_date')].min()).to_numpy().reshape(-1, 1)
    y = pd.to_numeric(g[C('cbs_cd4_count_results')], errors='coerce').to_numpy()
    return float(LinearRegression().fit(X, y).coef_[0])

if {C('cbs_cd4_collection_date'), C('cbs_cd4_count_results'), C('cbs_Index_UPID')}.issubset(df_no_last_test.columns):
    cd4_slope = df_no_last_test.groupby(C('cbs_Index_UPID')).apply(_safe_cd4_slope)
    delta_cd4_recent = df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_cd4_count_results')].apply(
        lambda x: x.dropna().diff().iloc[-1] if x.dropna().shape[0] >= 2 else np.nan
    )
    df_last_test['cd4_slope']        = df_last_test[C('cbs_Index_UPID')].map(cd4_slope.to_dict())
    df_last_test['delta_cd4_recent'] = df_last_test[C('cbs_Index_UPID')].map(delta_cd4_recent.to_dict())

# ---- Delays/turnaround from history ----
if {C('cbs_Date_de_debut_des_ARVs'), C('cbs_date_linked'), C('cbs_Index_UPID')}.issubset(df.columns):
    link_to_art_days = dt_days(df[C('cbs_Date_de_debut_des_ARVs')], df[C('cbs_date_linked')])
    df_last_test['link_to_art_days'] = (
        link_to_art_days.groupby(df[C('cbs_Index_UPID')]).last()
        .reindex(df_last_test[C('cbs_Index_UPID')]).values
    )

if {C('Report_date'), C('cbs_last_pharmacy_pick_up_date'), C('cbs_Index_UPID')}.issubset(df_no_last_test.columns):
    dslp_hist = dt_days(df_no_last_test[C('Report_date')], df_no_last_test[C('cbs_last_pharmacy_pick_up_date')])
    dslp_last = dslp_hist.groupby(df_no_last_test[C('cbs_Index_UPID')]).last()
    df_last_test['days_since_last_pickup'] = df_last_test[C('cbs_Index_UPID')].map(dslp_last.to_dict())

if {C('cbs_cd4_result_return_date'), C('cbs_cd4_collection_date'), C('cbs_Index_UPID')}.issubset(df_no_last_test.columns):
    cd4_tat_hist = dt_days(df_no_last_test[C('cbs_cd4_result_return_date')], df_no_last_test[C('cbs_cd4_collection_date')])
    cd4_tat_last = cd4_tat_hist.groupby(df_no_last_test[C('cbs_Index_UPID')]).last()
    df_last_test['cd4_turnaround'] = df_last_test[C('cbs_Index_UPID')].map(cd4_tat_last.to_dict())

if {C('cbs_Viral_load_result_return_date'), C('cbs_Viral_load_collection_date'), C('cbs_Index_UPID')}.issubset(df_no_last_test.columns):
    vl_tat_hist = dt_days(df_no_last_test[C('cbs_Viral_load_result_return_date')], df_no_last_test[C('cbs_Viral_load_collection_date')])
    vl_tat_last = vl_tat_hist.groupby(df_no_last_test[C('cbs_Index_UPID')]).last()
    df_last_test['vl_turnaround'] = df_last_test[C('cbs_Index_UPID')].map(vl_tat_last.to_dict())

# ---- Counts (historical) ----
if C('cbs_OI_Name') in df_no_last_test.columns:
    oi_count = df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_OI_Name')].nunique()
    df_last_test['oi_count'] = df_last_test[C('cbs_Index_UPID')].map(oi_count.to_dict())

if C('cbs_Change_in_regimen') in df_no_last_test.columns:
    reg_change_cnt = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Change_in_regimen')]
        .apply(lambda s: pd.to_numeric(s, errors='coerce').fillna(0).astype(int).sum())
    )
    df_last_test['regimen_change_count'] = df_last_test[C('cbs_Index_UPID')].map(reg_change_cnt.to_dict())

# ---- Binary risk flags ----
if C('cbs_Number_of_sexual_partners_in_last_3_months') in df.columns:
    last_3mo = df.groupby(C('cbs_Index_UPID'))[C('cbs_Number_of_sexual_partners_in_last_3_months')].last()
    df_last_test['many_partners_3mo'] = df_last_test[C('cbs_Index_UPID')].map((last_3mo > 3).astype(int).to_dict())

if C('cbs_Number_of_lifetime_sexual_partners') in df.columns:
    last_life = df.groupby(C('cbs_Index_UPID'))[C('cbs_Number_of_lifetime_sexual_partners')].last()
    df_last_test['many_partners_life'] = df_last_test[C('cbs_Index_UPID')].map((last_life > 10).astype(int).to_dict())

# ---- TPT flags from history only ----
tpt_flag_col = next((C(c) for c in ['cbs_initiated_on_tpt','cbs_Initiated_on_TPT'] if C(c) in df_no_last_test.columns), None)
if tpt_flag_col:
    on_tpt_last = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[tpt_flag_col]
        .last().astype(str).str.lower().map({'yes':1,'no':0})
    )
    df_last_test['on_tpt'] = df_last_test[C('cbs_Index_UPID')].map(on_tpt_last.to_dict()).fillna(0).astype(int)

if {C('cbs_if_yes_tpt_start_date'), C('cbs_Date_de_debut_des_ARVs'), C('cbs_Index_UPID')}.issubset(df.columns):
    tpt_duration = dt_days(df[C('cbs_if_yes_tpt_start_date')], df[C('cbs_Date_de_debut_des_ARVs')])
    df_last_test['tpt_duration'] = (
        tpt_duration.groupby(df[C('cbs_Index_UPID')]).last()
        .reindex(df_last_test[C('cbs_Index_UPID')]).values
    )

# ---- Regimen change flag BEFORE last test ----
if C('cbs_Change_in_regimen') in df_no_last_test.columns:
    last_change_hist = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Change_in_regimen')]
        .last().fillna(0).astype(int)
    )
    df_last_test['regimen_change_flag'] = df_last_test[C('cbs_Index_UPID')].map(last_change_hist.to_dict())

# ---- WHO stage (history only) ----
# WHO features restricted to info available up to last VL date

if {C('cbs_WHO_Stage'), C('cbs_WHO_stage_date')}.issubset(df.columns):
    stage_map = {'i':1,'ii':2,'iii':3,'iv':4}

    # Build WHO table from HISTORY rows in df (any case handled by C())
    who = df[[C('cbs_Index_UPID'), C('cbs_WHO_Stage'), C('cbs_WHO_stage_date')]].copy()
    who['who_num'] = who[C('cbs_WHO_Stage')].astype(str).str.lower().map(stage_map)

    # ⬇️ Define last_vl from df_last_test, keyed by UPID, then merge
    last_vl = (
        df_last_test
        .set_index(C('cbs_Index_UPID'))['Last_Viral_Load_Test_Date']
        .rename('last_vl')
    )
    who = who.merge(last_vl, left_on=C('cbs_Index_UPID'), right_index=True, how='left')

    # Keep WHO observations at or before the last VL test
    who = who[pd.to_datetime(who[C('cbs_WHO_stage_date')]) <= who['last_vl']]

    # Map back to df_last_test
    who_num_last = (who.sort_values(C('cbs_WHO_stage_date'))
                      .groupby(C('cbs_Index_UPID'))['who_num'].last())
    df_last_test['who_stage_num'] = df_last_test[C('cbs_Index_UPID')].map(who_num_last.to_dict())

    who_change = (who.sort_values(C('cbs_WHO_stage_date'))
                    .groupby(C('cbs_Index_UPID'))['who_num']
                    .apply(lambda s: s.ffill().bfill().diff().fillna(0).astype(int).iloc[-1] if len(s) > 1 else 0))
    df_last_test['who_stage_change'] = df_last_test[C('cbs_Index_UPID')].map(who_change.to_dict())

    
# ---- Days since enrollment (history only) ----
if {C('Report_date'), C('enrollment_date_in_cbs')}.issubset(df_no_last_test.columns):
    dse_hist = dt_days(df_no_last_test[C('Report_date')], df_no_last_test[C('enrollment_date_in_cbs')])
    dse_last = dse_hist.groupby(df_no_last_test[C('cbs_Index_UPID')]).last()
    df_last_test['days_since_enrollment'] = df_last_test[C('cbs_Index_UPID')].map(dse_last.to_dict())

# ---- Adherence variability (history only) ----
if C('cbs_Client_adherence') in df_no_last_test.columns:
    adh_num = df_no_last_test[C('cbs_Client_adherence')].replace({'good':2,'moderate':1,'bad':0})
    adh_sd = adh_num.groupby(df_no_last_test[C('cbs_Index_UPID')]).rolling(3).std().reset_index(level=0, drop=True)
    adh_sd_last = adh_sd.groupby(df_no_last_test[C('cbs_Index_UPID')]).last()
    df_last_test['adherence_sd_3visits'] = df_last_test[C('cbs_Index_UPID')].map(adh_sd_last.to_dict())

# ---- Visit frequency (history only) ----
if C('cbs_Client_visit_date') in df_no_last_test.columns:
    def _count_last_year(dates):
        dates = pd.to_datetime(dates).sort_values()
        return dates.map(lambda d: dates[(dates >= d - pd.Timedelta(days=365)) & (dates <= d)].count())
    visits_last_12mo = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Client_visit_date')]
        .apply(_count_last_year).groupby(level=0).last()
    )
    df_last_test['visits_last_12mo'] = df_last_test[C('cbs_Index_UPID')].map(visits_last_12mo.to_dict())
    df_last_test['missed_visit_rate'] = ((12 - df_last_test['visits_last_12mo']) / 12).clip(lower=0, upper=1)

# ---- Time-weighted suppression (history only) ----
if {C('cbs_Viral_Load'), C('cbs_Viral_load_collection_date'),
    C('Report_date'), C('cbs_Index_UPID')}.issubset(df_no_last_test.columns):
    ref_date = pd.to_datetime(df[C('Report_date')].max())
    days_since = dt_days(ref_date, df_no_last_test[C('cbs_Viral_load_collection_date')])
    decay_wt   = np.exp(-days_since / 180.0)
    suppr_ind  = (pd.to_numeric(df_no_last_test[C('cbs_Viral_Load')], errors='coerce') < 200).astype(int)
    tw = (pd.DataFrame({
            C('cbs_Index_UPID'): df_no_last_test[C('cbs_Index_UPID')],
            'suppr_ind': suppr_ind, 'decay_wt': decay_wt
         }).groupby(C('cbs_Index_UPID'))
         .apply(lambda g: (g['suppr_ind'] * g['decay_wt']).sum() / g['decay_wt'].sum()
               if g['decay_wt'].sum() > 0 else np.nan))
    df_last_test['tw_suppression'] = df_last_test[C('cbs_Index_UPID')].map(tw.to_dict())

# ---- Seasonality (use test month; no outcome info) ----
df_last_test['rainy_season'] = df_last_test['Last_Viral_Load_Test_Date'].dt.month.isin([4,5,6,10,11,12]).astype(int)

# ---- Pregnancy & parity (history only) ----
if C('cbs_pregnant') in df_no_last_test.columns:
    preg_last = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_pregnant')]
        .last().astype(str).str.lower().map({'yes':1,'no':0})
    )
    df_last_test['pregnancy_flag'] = df_last_test[C('cbs_Index_UPID')].map(preg_last.to_dict()).fillna(0).astype(int)
    if C('cbs_Client_adherence') in df_no_last_test.columns:
        adh_map = {'good':2,'moderate':1,'bad':0}
        adh_num_last = (
            df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Client_adherence')]
            .last().astype(str).str.lower().map(adh_map)
        )
        df_last_test['pregnancy_adherence_int'] = (
            df_last_test['pregnancy_flag'] * df_last_test[C('cbs_Index_UPID')].map(adh_num_last.to_dict())
        )

if C('num_alive_children_under_15') in df.columns and C('cbs_Client_visit_date') in df_no_last_test.columns:
    gap_last = df_no_last_test.groupby(C('cbs_Index_UPID')).apply(
        lambda g: dt_days(g[C('cbs_Client_visit_date')], g[C('cbs_Client_visit_date')].shift(1)).iloc[-1]
        if g.shape[0] >= 2 else np.nan
    )
    par_last = df.groupby(C('cbs_Index_UPID'))[C('num_alive_children_under_15')].last()
    df_last_test['parity_gap_int'] = (
        df_last_test[C('cbs_Index_UPID')].map(par_last.to_dict()) *
        df_last_test[C('cbs_Index_UPID')].map(gap_last.to_dict())
    )

if {'current_age','pregnancy_flag'}.issubset(df_last_test.columns):
    df_last_test['young_pregnant'] = ((df_last_test['current_age'] < 25) & (df_last_test['pregnancy_flag'] == 1)).astype(int)

# ---- Suppression history / adherence trend (history only) ----
if C('cbs_Viral_Load') in df_no_last_test.columns:
    df_no_last_test['Suppressed_Counts'] = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Viral_Load')]
        .transform(lambda x: x.shift().expanding().apply(lambda v: np.sum(pd.to_numeric(v, errors='coerce') < 200), raw=True))
        .fillna(0).astype(int)
    )
else:
    df_no_last_test['Suppressed_Counts'] = np.nan

if C('cbs_Client_adherence') in df_no_last_test.columns:
    df_no_last_test['Adherence_Trend'] = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Client_adherence')]
        .apply(lambda s: s.replace({'good':2,'moderate':1,'bad':0}).astype(float).diff())
        .reset_index(level=0, drop=True)
    )
else:
    df_no_last_test['Adherence_Trend'] = np.nan

# ---- Visit gaps (history only) ----
if C('cbs_Client_visit_date') in df_no_last_test.columns:
    df_no_last_test['Prev_Visit_Date'] = (
        df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Client_visit_date')].shift(1)
    )
    df_no_last_test['Days_Between_Visits'] = dt_days(
        df_no_last_test[C('cbs_Client_visit_date')], df_no_last_test['Prev_Visit_Date']
    )
    avg_gap = df_no_last_test.groupby(C('cbs_Index_UPID'))['Days_Between_Visits'].mean()
    max_gap = df_no_last_test.groupby(C('cbs_Index_UPID'))['Days_Between_Visits'].max()
    df_last_test['Avg_Visit_Gap'] = df_last_test[C('cbs_Index_UPID')].map(avg_gap.to_dict())
    df_last_test['Max_Visit_Gap'] = df_last_test[C('cbs_Index_UPID')].map(max_gap.to_dict())

# ---- Previous VL & dynamics (history only) ----
if C('cbs_Viral_Load') in df_no_last_test.columns:
    prev_vl = df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Viral_Load')].last()
    df_last_test['Previous_VL'] = df_last_test[C('cbs_Index_UPID')].map(prev_vl.to_dict())

def compute_delta_vl(group):
    vls = pd.to_numeric(group[C('cbs_Viral_Load')], errors='coerce').dropna().to_numpy()
    placeholder = -999
    if len(vls) >= 3:
        delta_recent = float(vls[-2]) - float(vls[-3])
        delta_long   = float(vls[-2]) - float(vls[0])
    elif len(vls) >= 2:
        delta_recent = placeholder
        delta_long   = float(vls[-1]) - float(vls[0])
    else:
        delta_recent = placeholder
        delta_long   = placeholder
    return pd.Series({'Delta_VL_recent': delta_recent, 'Delta_VL_long': delta_long})

if C('cbs_Viral_Load') in df_no_last_test.columns:
    delta_vl_stats = safe_reset_index(
        df_no_last_test.groupby(C('cbs_Index_UPID')).apply(compute_delta_vl)
    )
    df_last_test = df_last_test.merge(delta_vl_stats, left_on=C('cbs_Index_UPID'), right_on=C('cbs_Index_UPID'), how='left')

# ---- Risk score (history only) ----
risk_vars = [
    C('cbs_Any_Changes_in_or_Undisclosed_Sexual_Partners'),
    C('cbs_Sex_with_multiple_partners'),
    C('cbs_Sex_with_commercial_sex_worker'),
    C('cbs_Any_vaginal_or_anal_sex_without_condoms_in_last_12_months')
]
present_risk = [c for c in risk_vars if c in df_no_last_test.columns]
if present_risk:
    for col in present_risk:
        df_no_last_test[col] = pd.to_numeric(df_no_last_test[col], errors='coerce').fillna(0)
    risk_scores = df_no_last_test.groupby(C('cbs_Index_UPID'))[present_risk].max().sum(axis=1)
    df_last_test['Risk_Score'] = df_last_test[C('cbs_Index_UPID')].map(risk_scores.to_dict())

# ---- Late ART initiation (baseline; safe) ----
if {C('cbs_Index_UPID'), 'Days_HIV_Diagnosis_to_ART'}.issubset(df.columns):
    diagnosis_to_art = df.groupby(C('cbs_Index_UPID'))['Days_HIV_Diagnosis_to_ART'].first()
    df_last_test['Days_HIV_Diagnosis_to_ART'] = df_last_test[C('cbs_Index_UPID')].map(diagnosis_to_art.to_dict())
    df_last_test['Late_ART_Initiation'] = (df_last_test['Days_HIV_Diagnosis_to_ART'] > 15).astype(int)

# ---- VL slope & switches (history only) ----
if {C('cbs_Viral_load_collection_date'), C('cbs_Viral_Load')}.issubset(df_no_last_test.columns):
    vl_slope = df_no_last_test.groupby(C('cbs_Index_UPID')).apply(
        lambda g: float(LinearRegression().fit(
            dt_days(g[C('cbs_Viral_load_collection_date')], g[C('cbs_Viral_load_collection_date')].min()).to_numpy().reshape(-1, 1),
            pd.to_numeric(g[C('cbs_Viral_Load')], errors='coerce').to_numpy()
        ).coef_[0]) if g.shape[0] > 1 else 0.0
    )
    df_last_test['VL_Slope'] = df_last_test[C('cbs_Index_UPID')].map(vl_slope.to_dict())

if C('cbs_Viral_Load') in df_no_last_test.columns:
    def count_switches(series):
        series = pd.to_numeric(series, errors='coerce')
        return (((series.shift() < 200) & (series >= 200)) | ((series.shift() >= 200) & (series < 200))).sum()
    vl_switches = df_no_last_test.groupby(C('cbs_Index_UPID'))[C('cbs_Viral_Load')].apply(count_switches)
    df_last_test['VL_Switches'] = df_last_test[C('cbs_Index_UPID')].map(vl_switches.to_dict())

# ---- Historical summary to last-test row ----
if {'Suppressed_Counts','Adherence_Trend'}.issubset(df_no_last_test.columns):
    hist_summary = safe_reset_index(
        df_no_last_test.groupby(C('cbs_Index_UPID'))[['Suppressed_Counts', 'Adherence_Trend']].last()
    )
    df_last_test = df_last_test.merge(hist_summary, left_on=C('cbs_Index_UPID'), right_on=C('cbs_Index_UPID'), how='left')

# ──────────────────────────────────────────────────────────────────────────────
# Outcome history features (HISTORY ONLY)  ➜ prev_outcome limited to {on_art, ltfu}
# ──────────────────────────────────────────────────────────────────────────────
out_col = C('cbs_Client_outcome')
upid = C('cbs_Index_UPID')
outcome_feats = []  # collected names

if out_col in df_no_last_test.columns:
    def _norm_outcome_binary(s: pd.Series) -> pd.Series:
        s = s.astype(str).str.strip().str.lower()
        s = s.replace({
            'alive and on art': 'on_art',
            'alive & on art':   'on_art',
            'on art':           'on_art',
            'active on art':    'on_art',
            'on arv':           'on_art',
            'on arv treatment': 'on_art',
            'restarted on art': 'on_art',
            'ltfu':               'ltfu',
            'lost to follow-up':  'ltfu',
            'lost to follow up':  'ltfu',
            'lost to followup':   'ltfu',
            'lost to follow up (ltfu)': 'ltfu',
        })
        s = s.replace({'nan': np.nan, 'none': np.nan, '': np.nan})
        keep = s.isin(['on_art', 'ltfu'])
        s.loc[~keep] = np.nan  # drop died/transfer_out/etc from prev_outcome
        return s

    df_no_last_test['outcome_norm'] = _norm_outcome_binary(df_no_last_test[out_col])

    # Choose a date anchor present in HISTORY
    anchor = None
    if C('Report_date') in df_no_last_test.columns:
        anchor = C('Report_date')
    elif C('cbs_Client_visit_date') in df_no_last_test.columns:
        anchor = C('cbs_Client_visit_date')

    # 1) Last historical outcome (one-hot limited to {on_art, ltfu} + unknown)
    prev_out = df_no_last_test.groupby(upid)['outcome_norm'].last()
    df_last_test['prev_outcome'] = df_last_test[upid].map(prev_out.to_dict())
    for cat in ['on_art', 'ltfu']:
        df_last_test[f'prev_outcome_{cat}'] = (df_last_test['prev_outcome'] == cat).astype(int)
    df_last_test['prev_outcome_unknown'] = df_last_test['prev_outcome'].isna().astype(int)

    # 2) Ever LTFU before (binary)
    ever_ltfu = df_no_last_test.groupby(upid)['outcome_norm'].apply(lambda s: (s == 'ltfu').any()).astype(int)
    df_last_test['ever_ltfu_before'] = df_last_test[upid].map(ever_ltfu.to_dict()).fillna(0).astype(int)

    # Helpers
    def _switches(s):
        s = s.dropna()
        return int((s != s.shift()).sum() - 1) if len(s) > 1 else 0

    def _episodes(s, state='ltfu'):
        m = s.eq(state).astype(int)
        return int(((m.shift(fill_value=0) == 0) & (m == 1)).sum())

    def _longest_streak(s, state='ltfu'):
        m = s.eq(state).astype(int)
        if m.sum() == 0:
            return 0
        grp = (m != m.shift(fill_value=0)).cumsum()
        return int(m[m == 1].groupby(grp).size().max())

    # 3) Outcome dynamics
    switches        = df_no_last_test.groupby(upid)['outcome_norm'].apply(_switches)
    ltfu_episodes   = df_no_last_test.groupby(upid)['outcome_norm'].apply(lambda s: _episodes(s, 'ltfu'))
    onart_longest   = df_no_last_test.groupby(upid)['outcome_norm'].apply(lambda s: _longest_streak(s, 'on_art'))
    ltfu_longest    = df_no_last_test.groupby(upid)['outcome_norm'].apply(lambda s: _longest_streak(s, 'ltfu'))

    df_last_test['outcome_switches']     = df_last_test[upid].map(switches.to_dict())
    df_last_test['ltfu_episode_count']   = df_last_test[upid].map(ltfu_episodes.to_dict())
    df_last_test['longest_onart_streak'] = df_last_test[upid].map(onart_longest.to_dict())
    df_last_test['longest_ltfu_streak']  = df_last_test[upid].map(ltfu_longest.to_dict())

    # 4) Time-decayed share of "on ART" in history
    if anchor is not None:
        last_hist = df_no_last_test.groupby(upid)[anchor].max()
        days = dt_days(last_hist.reindex(df_no_last_test[upid]).values, df_no_last_test[anchor])
        w = np.exp(-days / 180.0)
        onart_ind = (df_no_last_test['outcome_norm'] == 'on_art').astype(int)
        tw_onart = (pd.DataFrame({upid: df_no_last_test[upid], 'w': w, 'x': onart_ind})
                      .groupby(upid)
                      .apply(lambda g: (g['w'] * g['x']).sum() / g['w'].sum() if g['w'].sum() > 0 else np.nan))
        df_last_test['tw_onart_share'] = df_last_test[upid].map(tw_onart.to_dict())

    # 5) Recency: days since last LTFU (relative to last historical anchor date)
    if anchor is not None:
        last_hist_date = df_no_last_test.groupby(upid)[anchor].max()
        last_ltfu_date = (
            df_no_last_test.loc[df_no_last_test['outcome_norm'] == 'ltfu']
            .groupby(upid)[anchor]
            .max()
        )
        tmp2 = pd.DataFrame({
            'last_hist': pd.to_datetime(last_hist_date, errors='coerce'),
            'last_ltfu': pd.to_datetime(last_ltfu_date, errors='coerce'),
        })
        tmp2['days_since_last_ltfu'] = (tmp2['last_hist'] - tmp2['last_ltfu']).dt.days
        df_last_test['days_since_last_ltfu'] = df_last_test[upid].map(tmp2['days_since_last_ltfu'].to_dict())

    # 6) Return from LTFU + time to return
    if anchor is not None:
        def _return_info(g):
            s = g['outcome_norm']
            if not (s == 'ltfu').any():
                return pd.Series({'returned_from_ltfu': 0, 'days_to_return_from_last_ltfu': np.nan})
            idx_last_ltfu = s[s == 'ltfu'].index.max()
            after = g.loc[g.index > idx_last_ltfu]
            if after.empty:
                return pd.Series({'returned_from_ltfu': 0, 'days_to_return_from_last_ltfu': np.nan})
            ret_rows = after.loc[after['outcome_norm'] == 'on_art']
            if ret_rows.empty:
                return pd.Series({'returned_from_ltfu': 0, 'days_to_return_from_last_ltfu': np.nan})
            d = dt_days(ret_rows[anchor].iloc[0], g.loc[idx_last_ltfu, anchor])
            return pd.Series({'returned_from_ltfu': 1, 'days_to_return_from_last_ltfu': float(d)})

        rinfo = df_no_last_test.groupby(upid)[['outcome_norm', anchor]].apply(_return_info)
        df_last_test['returned_from_ltfu'] = df_last_test[upid].map(rinfo['returned_from_ltfu'].to_dict()).fillna(0).astype(int)
        df_last_test['days_to_return_from_last_ltfu'] = df_last_test[upid].map(rinfo['days_to_return_from_last_ltfu'].to_dict())

    # 7) Early LTFU within 180 days of ART start
    if (C('cbs_Date_de_debut_des_ARVs') in df.columns) and (anchor is not None):
        art_start = df.groupby(upid)[C('cbs_Date_de_debut_des_ARVs')].first()
        first_ltfu_date = (
            df_no_last_test.loc[df_no_last_test['outcome_norm'] == 'ltfu']
            .groupby(upid)[anchor]
            .min()
        )
        tmp = pd.DataFrame({
            'art_start':  pd.to_datetime(art_start, errors='coerce'),
            'first_ltfu': pd.to_datetime(first_ltfu_date, errors='coerce'),
        })
        tmp['days_to_first_ltfu'] = (tmp['first_ltfu'] - tmp['art_start']).dt.days.astype('float64')
        early_flag = (tmp['days_to_first_ltfu'] <= 180).fillna(False).astype(int)
        df_last_test['early_ltfu_180d'] = df_last_test[upid].map(early_flag.to_dict()).fillna(0).astype(int)

    # 8) Transition counts: A→L and L→A
    def _trans_counts(s):
        s = s.dropna()
        if len(s) < 2:
            return pd.Series({'A_to_L': 0, 'L_to_A': 0})
        prev = s.shift()
        return pd.Series({
            'A_to_L': int(((prev == 'on_art') & (s == 'ltfu')).sum()),
            'L_to_A': int(((prev == 'ltfu') & (s == 'on_art')).sum()),
        })
    trans = df_no_last_test.groupby(upid)['outcome_norm'].apply(_trans_counts).unstack(fill_value=0)
    for col in ['A_to_L', 'L_to_A']:
        if col in trans.columns:
            df_last_test[col] = df_last_test[upid].map(trans[col].to_dict())

    # 9) Outcome variability (Shannon entropy)
    def _entropy(s):
        vc = s.value_counts(dropna=True)
        if vc.sum() == 0:
            return np.nan
        p = vc / vc.sum()
        return float(-(p * np.log2(p)).sum())
    ent = df_no_last_test.groupby(upid)['outcome_norm'].apply(_entropy)
    df_last_test['outcome_entropy'] = df_last_test[upid].map(ent.to_dict())

    # register names created in this block
    outcome_feats = [
        'prev_outcome','prev_outcome_on_art','prev_outcome_ltfu','prev_outcome_unknown',
        'ever_ltfu_before','outcome_switches','ltfu_episode_count','longest_onart_streak',
        'longest_ltfu_streak','tw_onart_share','days_since_last_ltfu','returned_from_ltfu',
        'days_to_return_from_last_ltfu','early_ltfu_180d','A_to_L','L_to_A','outcome_entropy'
    ]
    
# ──────────────────────────────────────────────────────────────────────────────
# 4) Merge back cleanly (single source of truth from df_last_test)
# ──────────────────────────────────────────────────────────────────────────────
# DO NOT include post-outcome features (no VL_Category, no days_since_last_vl/last_vl_bucket, no month)
computed_base = [
    'Viral_Load',                     # <-- label (keep in df; drop when modeling X)
    'Avg_Visit_Gap', 'Max_Visit_Gap', 'Previous_VL',
    'Delta_VL_recent', 'Delta_VL_long', 'Risk_Score', 'Late_ART_Initiation',
    'VL_Slope', 'VL_Switches', 'Suppressed_Counts', 'Adherence_Trend'
]
extra_features = [
    'current_age','Num_VL_Tests','Num_Suppressed','Suppression_Rate',
    'cd4_slope','delta_cd4_recent','link_to_art_days','days_since_last_pickup',
    'cd4_turnaround','vl_turnaround','oi_count','regimen_change_count',
    'many_partners_3mo','many_partners_life','on_tpt','tpt_duration','regimen_change_flag','who_stage_num',
    'days_since_enrollment','who_stage_change','adherence_sd_3visits','visits_last_12mo',
    'missed_visit_rate','tw_suppression','rainy_season',
    'pregnancy_flag','pregnancy_adherence_int','parity_gap_int','young_pregnant'
]
        
# Features to carry from df_last_test back to df
computed_features = list(dict.fromkeys(
    computed_base
    + [f for f in extra_features   if f in df_last_test.columns]
    + [f for f in outcome_feats    if f in df_last_test.columns]  # <-- add outcome features
))

# Ensure any missing columns exist (so merge/coalesce logic is stable)
for c in computed_features:
    if c not in df_last_test.columns:
        df_last_test[c] = np.nan


# remove any stale copies in df to avoid _x/_y on merge
to_drop_in_df = [c for c in computed_features if c in df.columns]
if to_drop_in_df:
    df.drop(columns=to_drop_in_df, inplace=True, errors='ignore')
if '_merge' in df.columns:
    df.drop(columns='_merge', inplace=True, errors='ignore')

# IMPORTANT: also drop leakage-prone raw column if it exists
# (cumsum across full df could include reactions at last row)
if 'Num_ART_Regimen_Changes' in df.columns:
    df.drop(columns=['Num_ART_Regimen_Changes'], inplace=True, errors='ignore')

df = df.merge(
    df_last_test[[C('cbs_Index_UPID')] + computed_features],
    left_on=C('cbs_Index_UPID'), right_on=C('cbs_Index_UPID'),
    how='left', validate='m:1'
)

# coalesce potential *_x/_y (safety net)
for base in computed_features:
    x, y = f"{base}_x", f"{base}_y"
    if x in df.columns or y in df.columns:
        df[base] = df[y] if y in df.columns else np.nan
        if x in df.columns:
            df[base] = df[base].combine_first(df[x])
        for col in (x, y):
            if col in df.columns:
                df.drop(columns=col, inplace=True, errors='ignore')

# ──────────────────────────────────────────────────────────────────────────────
# 5) Final derived metrics from HISTORY (no leakage)
# ──────────────────────────────────────────────────────────────────────────────
def calculate_days_to_first_suppression(df_all, base):
    art_start = base.groupby(C('cbs_Index_UPID'))[C('cbs_Date_de_debut_des_ARVs')].first()
    first_supp = (
        base.loc[pd.to_numeric(base[C('cbs_Viral_Load')], errors='coerce') < 200]
            .groupby(C('cbs_Index_UPID'))[C('cbs_Viral_load_collection_date')]
            .min()
    )
    days = dt_days(first_supp, art_start)
    df_all['Days_to_First_Suppression'] = df_all[C('cbs_Index_UPID')].map(days.to_dict())
    return df_all

def calc_effective_suppression_time(df_all, base):
    base = base.dropna(subset=[C('cbs_Viral_load_collection_date')]).sort_values(
        [C('cbs_Index_UPID'), C('cbs_Viral_load_collection_date')]
    )
    def eff(group):
        g = group
        v = pd.to_numeric(g[C('cbs_Viral_Load')], errors='coerce')
        mask = v < 200
        if not mask.any():
            return np.nan
        first_date = g.loc[mask, C('cbs_Viral_load_collection_date')].iloc[0]
        after = g[g[C('cbs_Viral_load_collection_date')] >= first_date]
        rebound = after.loc[pd.to_numeric(after[C('cbs_Viral_Load')], errors='coerce') >= 200]
        end_date = (rebound[C('cbs_Viral_load_collection_date')].iloc[0] 
                    if not rebound.empty 
                    else after[C('cbs_Viral_load_collection_date')].iloc[-1])
        return int(dt_days(end_date, first_date))
    eff_days = base.groupby(C('cbs_Index_UPID'), as_index=True).apply(eff)
    df_all['Effective_Suppression_Time'] = df_all[C('cbs_Index_UPID')].map(eff_days.to_dict())
    return df_all

df = calculate_days_to_first_suppression(df, df_no_last_test)
df = calc_effective_suppression_time(df, df_no_last_test)

# Derived flag (safe)
df['delayed_suppression'] = (df['Days_to_First_Suppression'] > 0).astype(int)

# Fill placeholders for downstream numeric use
df['Days_to_First_Suppression']  = df['Days_to_First_Suppression'].fillna(-1)
df['Effective_Suppression_Time'] = df['Effective_Suppression_Time'].fillna(0)

# Final deduplication by client
df = df.drop_duplicates(subset=C('cbs_Index_UPID'), keep='last')

print(f"✅ Final dataset shape: {df.shape}")
print("✅ Data preprocessing completed successfully — history-only features where needed (no leakage).")


## Exploratory Data Analysis

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
### function to plot distributions wrt target


def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
        stat="density",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
        stat="density",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
def stacked_barplot(data, predictor, target, n=None):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    n: number of top categories to display
    """
    if n is not None:
        top_categories = data[predictor].value_counts().index[:n]
        data = data[data[predictor].isin(top_categories)]

    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 5, 6))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
# Create a list of columns that should be excluded from categorical/numerical plotting
# These are usually ID columns or merge-checking columns, not useful for bar plots
exclude_cols = [
    'cbs_Index UPID',
    '_merge'
]

# Keep only excluded columns that actually exist in df
# This prevents errors if one of the listed columns is not found
exclude_cols = [col for col in exclude_cols if col in df.columns]


# Create categorical column list
# This selects columns with text/object, category, or boolean data types
cat_col = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Remove ID or unwanted columns from the categorical list
cat_col = [col for col in cat_col if col not in exclude_cols]


# Create numerical column list
# This selects columns with numeric data types such as int, float, and nullable Int64
num_col = df.select_dtypes(include=['number']).columns.tolist()

# Remove ID or unwanted columns from the numerical list
num_col = [col for col in num_col if col not in exclude_cols]


# Some numeric columns may actually be categorical, for example 0/1 variables
# We identify numeric columns with few unique values and move them to cat_col
low_cardinality_num = [
    col for col in num_col
    if df[col].nunique(dropna=True) <= 10
]

# Add low-cardinality numeric columns to categorical columns
cat_col = cat_col + low_cardinality_num

# Remove those low-cardinality columns from numerical columns
num_col = [col for col in num_col if col not in low_cardinality_num]


# Print the number of categorical and numerical columns
print("Number of categorical columns:", len(cat_col))
print("Number of numerical columns:", len(num_col))

# Display categorical columns
print("\nCategorical columns:")
print(cat_col)

# Display numerical columns
print("\nNumerical columns:")
print(num_col)

In [ ]:
# Loop through each categorical variable in cat_col
for var in cat_col:
    
    # Generate a labeled bar plot for each categorical variable
    labeled_barplot(df, var, perc=True)

In [ ]:
# Loop through each numerical variable in num_col
for var in num_col:
    
    # Generate a histogram and boxplot for each numerical variable
    histogram_boxplot(
        df,
        var,
        figsize=(10, 7),
        kde=True,
        bins=None
    )

## Target Analysis

In [ ]:
distribution_plot_wrt_target(df, 'cbs_CD4_count_results', 'Viral_Load')

In [ ]:
distribution_plot_wrt_target(df, 'cbs_Sexual_partners_in_last_12_months', 'Viral_Load')

In [ ]:
distribution_plot_wrt_target(df_reset, 'days_since_HIV_confirmation', 'Viral_Load')

In [ ]:
stacked_barplot(df, 'cbs_Any_vaginal_or_anal_sex_without_condoms_in_last_12_months', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Current_ART_Regimen', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Current_residence_type', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'Province', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Has_been_commercial_sex_worker', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Index_Case_Type', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Initial_ART_regimen', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_OI_Name', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Sex_with_HIV+_person', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Sex_with_female', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_Sex_with_male', 'Viral_Load',n=None)

In [ ]:
stacked_barplot(df, 'cbs_gender', 'Viral_Load',n=None)

In [ ]:
df['current_age'].describe()

In [ ]:
import pandas as pd

# Define bins and labels
bins = [0, 24, 34, 49, float('inf')]
labels = ["18-24", "25-34", "35-49", "50+"]

# Create the new variable
df["age_range"] = pd.cut(
    df["current_age"],
    bins=bins,
    labels=labels,
    right=True,      # upper bound inclusive
    include_lowest=True
)

# Check results
df[["current_age", "age_range"]].head(10)

In [ ]:
# Step 2: Define X (features) and y (target)
X = df.drop(columns=['Viral_Load'])  # Drop target variable from features
y = df['Viral_Load']  # Target variable

# Step 3: Train-test split (before deduplication)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=15, stratify=y
)



# Check dataset shapes after split and deduplication
print("Training Set Shape:", X_train.shape, y_train.shape)
print("Test Set Shape:", X_test.shape, y_test.shape)


In [ ]:
%%time


import warnings, numpy as np

# silence numpy divide/invalid warnings
np.seterr(divide='ignore', invalid='ignore')

# silence only those from sklearn.utils.extmath
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module=r"sklearn\.utils\.extmath",
)

# (optional) extra-targeted by message text
warnings.filterwarnings(
    "ignore",
    message=r".*invalid value encountered in divide.*",
    category=RuntimeWarning,
)


from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler, Normalizer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBClassifier, XGBRegressor

# ---------- config ----------
n_neighbors_range = [5, 10, 15, 20]
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
    #'Normalizer': Normalizer()  # (row-wise L2); usable here because we don't inverse back
}
FIXED_K_FOR_SCALER_SEARCH = 10  # reasonable mid-point

# ---------- helpers ----------
def make_ohe():
    try:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)

def is_binary_target(y):
    vals = pd.Series(y).dropna().unique()
    return set(np.unique(vals)).issubset({0, 1}) and len(vals) <= 2

def build_preprocessor(scaler, n_neighbors, cat_cols, num_cols):
    num_pipe = Pipeline([
        ('scale', scaler),
        ('knn',   KNNImputer(n_neighbors=n_neighbors, weights='distance'))
    ])
    cat_pipe = Pipeline([
        ('mode', SimpleImputer(strategy='most_frequent')),
        ('ohe',  make_ohe())
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ], remainder='drop')

# NEW (put here, replacing any older version)
def feature_names_from(prep, cat_cols, num_cols):
    names = list(num_cols)
    cat_tf = prep.named_transformers_.get('cat', None)
    if hasattr(cat_tf, "named_steps") and 'ohe' in cat_tf.named_steps:
        ohe = cat_tf.named_steps['ohe']
        if hasattr(ohe, 'categories_'):
            try:
                names += list(ohe.get_feature_names_out(cat_cols))
            except Exception:
                pass
    return names


# ---------- split columns ----------
cat_cols = selector(dtype_include=['object','category'])(X_train)
num_cols = selector(dtype_exclude=['object','category'])(X_train)

# Handle categoricals that are all-NaN (mode imputer needs at least one category)
all_nan_cats = [c for c in cat_cols if X_train[c].notna().sum() == 0]
if all_nan_cats:
    X_train[all_nan_cats] = X_train[all_nan_cats].fillna('__ALL_MISSING__')
    X_test[all_nan_cats]  = X_test[all_nan_cats].fillna('__ALL_MISSING__')

# ---------- Step 1: choose scaler via Logistic Regression ----------
print("Searching best scaler with LogisticRegression (AUC)...")
best_scaler_name, best_scaler_score = None, -np.inf
for sc_name, sc in scalers.items():
    prep = build_preprocessor(sc, FIXED_K_FOR_SCALER_SEARCH, cat_cols, num_cols)
    pipe = Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(max_iter=1000, n_jobs=None, solver='lbfgs'))
    ])
    if is_binary_target(y_train):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'roc_auc'
    else:
        # If target is continuous, LR isn't appropriate; switch to R^2 for a quick heuristic.
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'r2'
    try:
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        mean_score = np.mean(scores)
        print(f"  {sc_name:>14s}  -> {scoring}={mean_score:.4f}")
        if mean_score > best_scaler_score:
            best_scaler_name = sc_name
            best_scaler_score = mean_score
    except Exception as e:
        print(f"  {sc_name:>14s}  -> SKIPPED ({e})")

print(f"✅ Best scaler: {best_scaler_name} ({best_scaler_score:.4f})\n")
chosen_scaler = scalers[best_scaler_name]

# ---------- Step 2: choose K via XGBoost (AUC for binary, RMSE for regression) ----------
print("Searching best n_neighbors with XGBoost...")
best_k, best_k_score = None, -np.inf if is_binary_target(y_train) else np.inf

for k in n_neighbors_range:
    prep = build_preprocessor(chosen_scaler, k, cat_cols, num_cols)
    if is_binary_target(y_train):
        model = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            n_jobs=-1, random_state=42, eval_metric='logloss'
        )
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'roc_auc'
    else:
        model = XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            n_jobs=-1, random_state=42, objective='reg:squarederror'
        )
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        scoring = 'neg_root_mean_squared_error'

    pipe = Pipeline([('prep', prep), ('model', model)])
    try:
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        mean_score = np.mean(scores)
        if is_binary_target(y_train):
            better = mean_score > best_k_score
        else:
            better = mean_score > best_k_score  # less negative RMSE is better (higher)
        print(f"  k={k:>2} -> {scoring}={mean_score:.5f}")
        if better:
            best_k = k
            best_k_score = mean_score
    except Exception as e:
        print(f"  k={k:>2} -> SKIPPED ({e})")

print(f"✅ Best k: {best_k} (score={best_k_score:.5f})\n")

# ---------- Step 3: apply chosen (scaler, k) to build final X matrices ----------

final_prep = build_preprocessor(chosen_scaler, best_k, cat_cols, num_cols)
final_prep.fit(X_train)

Xtr_arr = final_prep.transform(X_train)
Xte_arr = final_prep.transform(X_test)

feat_names = feature_names_from(final_prep, cat_cols, num_cols)

X_train = pd.DataFrame(Xtr_arr, columns=feat_names, index=X_train.index)
X_test  = pd.DataFrame(Xte_arr,  columns=feat_names, index=X_test.index)


print("🎯 Chosen:", best_scaler_name, "with k =", best_k)
print("X_train_final shape:", X_train.shape)
print("X_test_final  shape:", X_test.shape)
print("Remaining NaNs (train):", int(np.isnan(X_train.values).sum()))
print("Remaining NaNs (test) :", int(np.isnan(X_test.values).sum()))


In [ ]:
from verstack import PandasOptimizer

# For X_train
opt_train = PandasOptimizer()
X_train = opt_train.optimize_memory_usage(X_train)


# For X_test
opt_test = PandasOptimizer()
X_test = opt_test.optimize_memory_usage(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

def model_performance_classification_sklearn(model, predictors, target, threshold=0.5):
    """
    Compute Accuracy, Recall, Precision, F1, AUC for a classification model
    using a custom probability threshold.

    model: classifier
    predictors: independent variables (X)
    target: dependent variable (y)
    threshold: probability threshold for classification (default=0.5)
    """
    # Predict probabilities for positive class
    probs = model.predict_proba(predictors)[:, 1]

    # Apply threshold (>= for consistency)
    pred = (probs >= threshold).astype(int)

    # Compute metrics
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred, zero_division=0)
    precision = precision_score(target, pred, zero_division=0)
    f1 = f1_score(target, pred, zero_division=0)
    AUC = roc_auc_score(target, probs)

    # Put into DataFrame
    df_perf = pd.DataFrame(
        {
            "Accuracy": [acc],
            "Recall": [recall],
            "Precision": [precision],
            "F1": [f1],
            "AUC": [AUC]
        }
    ).round(3)

    return df_perf


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

def confusion_matrix_sklearn(model, predictors, target, threshold=0.5):
    """
    Plot confusion matrix with counts + percentages and print TN, FP, FN, TP.

    model: classifier
    predictors: independent variables (X)
    target: dependent variable (y)
    threshold: probability threshold for classification (default=0.5)
    """
    # Predict probabilities
    probs = model.predict_proba(predictors)[:, 1]

    # Apply threshold (>= for consistency)
    pred = (probs >= threshold).astype(int)

    # Confusion matrix
    cm = confusion_matrix(target, pred)
    tn, fp, fn, tp = cm.ravel()

    # Annotate with counts + percentages
    labels = np.array([
        [f"{tn}\n{tn/cm.sum():.2%}", f"{fp}\n{fp/cm.sum():.2%}"],
        [f"{fn}\n{fn/cm.sum():.2%}", f"{tp}\n{tp/cm.sum():.2%}"]
    ])

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="", cmap="magma_r", cbar=True,
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["True 0", "True 1"])
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.title(f"Confusion Matrix @ threshold={threshold}")

    # Print values clearly
    print(f"True Negative: {tn}")
    print(f"False Positive: {fp}")
    print(f"False Negative: {fn}")
    print(f"True Positive: {tp}")

    return cm


In [ ]:
from sklearn import metrics


# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))


results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# Convert input features to a supported data type
X_train = X_train.astype('float32')  # or 'float64'

# loop through all models to get the mean cross validated score
print("\n" "Cross-Validation performance on training dataset:" "\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(
        n_splits=10,n_repeats=5, random_state=1
    )  # Setting number of splits equal to 5
    cv_result = cross_val_score(
        estimator=model, X=X_train, y=y_train, scoring=scorer, cv=kfold
    )
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train, y_train)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

### Model Building with ovrersampled data

In [ ]:
print("Before OverSampling, counts of label '1': {}".format(sum(y_train == 1)))
print("Before OverSampling, counts of label '0': {} \n".format(sum(y_train == 0)))

# Synthetic Minority Over Sampling Technique
sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=1)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


print("After OverSampling, counts of label '1': {}".format(sum(y_train_over == 1)))
print("After OverSampling, counts of label '0': {} \n".format(sum(y_train_over == 0)))


print("After OverSampling, the shape of train_X: {}".format(X_train_over.shape))
print("After OverSampling, the shape of train_y: {} \n".format(y_train_over.shape))

In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))


results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_over, y=y_train_over, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_over, y_train_over)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

### Model Building with undersampled data

In [ ]:
# Random undersampler for under sampling the data
rus = RandomUnderSampler(random_state=1, sampling_strategy=1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)


print("Before UnderSampling, counts of label '1': {}".format(sum(y_train == 1)))
print("Before UnderSampling, counts of label '0': {} \n".format(sum(y_train == 0)))


print("After UnderSampling, counts of label '1': {}".format(sum(y_train_un == 1)))
print("After UnderSampling, counts of label '0': {} \n".format(sum(y_train_un == 0)))


print("After UnderSampling, the shape of train_X: {}".format(X_train_un.shape))
print("After UnderSampling, the shape of train_y: {} \n".format(y_train_un.shape))

In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_un, y=y_train_un, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_un, y_train_un)
    scores = f1_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

## SMOTETomek

In [ ]:
from imblearn.combine import SMOTETomek  # Hybrid Sampling: SMOTE + Tomek Links
from imblearn.under_sampling import EditedNearestNeighbours  # Alternative hybrid method

# Print class distribution before sampling
print(f"Before Hybrid Sampling, counts of label '1': {sum(y_train == 1)}")
print(f"Before Hybrid Sampling, counts of label '0': {sum(y_train == 0)} \n")

# Hybrid Sampling: SMOTE + Tomek Links
smt = SMOTETomek(sampling_strategy=1, random_state=1)  # Balancing classes
X_train_hybrid, y_train_hybrid = smt.fit_resample(X_train, y_train)

# Alternative: SMOTE + Edited Nearest Neighbors (Uncomment if you want this method)
# enn = EditedNearestNeighbours()
# X_train_hybrid, y_train_hybrid = enn.fit_resample(X_train_hybrid, y_train_hybrid)

# Print class distribution after Hybrid Sampling
print(f"After Hybrid Sampling, counts of label '1': {sum(y_train_hybrid == 1)}")
print(f"After Hybrid Sampling, counts of label '0': {sum(y_train_hybrid == 0)} \n")

# Print new dataset shape
print(f"After Hybrid Sampling, shape of train_X: {X_train_hybrid.shape}")
print(f"After Hybrid Sampling, shape of train_y: {y_train_hybrid.shape} \n")


In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_hybrid, y=y_train_hybrid, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_hybrid, y_train_hybrid)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

## Combining SMOTETomek & ENN



In [ ]:
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import EditedNearestNeighbours

# Step 1: Apply SMOTETomek (SMOTE + Tomek Links)
smt = SMOTETomek(sampling_strategy=1, random_state=1)
X_train_hybrid, y_train_hybrid = smt.fit_resample(X_train, y_train)

# Step 2: Apply EditedNearestNeighbours (ENN) to further clean noise
enn = EditedNearestNeighbours()
X_train_hybrid, y_train_hybrid = enn.fit_resample(X_train_hybrid, y_train_hybrid)

# Print final class distribution
print(f"After Hybrid Sampling (SMOTETomek + ENN), counts of label '1': {sum(y_train_hybrid == 1)}")
print(f"After Hybrid Sampling (SMOTETomek + ENN), counts of label '0': {sum(y_train_hybrid == 0)} \n")

# Print new dataset shape
print(f"After Hybrid Sampling (SMOTETomek + ENN), shape of train_X: {X_train_hybrid.shape}")
print(f"After Hybrid Sampling (SMOTETomek + ENN), shape of train_y: {y_train_hybrid.shape} \n")


In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_hybrid, y=y_train_hybrid, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_hybrid, y_train_hybrid)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

## EditedNearestNeighbours (ENN) 

In [ ]:
from imblearn.under_sampling import EditedNearestNeighbours

# Apply EditedNearestNeighbours (ENN) to remove misclassified majority class samples
enn = EditedNearestNeighbours()
X_train_final1, y_train_final1 = enn.fit_resample(X_train, y_train)

# Print class distribution after ENN
print(f"After ENN, counts of label '1': {sum(y_train_final1 == 1)}")
print(f"After ENN, counts of label '0': {sum(y_train_final1 == 0)} \n")

# Print dataset shape
print(f"After ENN, shape of train_X: {X_train_final1.shape}")
print(f"After ENN, shape of train_y: {y_train_final1.shape} \n")


In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_final1, y=y_train_final1, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_final1, y_train_final1)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

## SMOTE After ENN

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

# Step 1: Apply ENN (Undersampling - Remove Noisy Majority Samples)
enn = EditedNearestNeighbours()
X_train_enn, y_train_enn = enn.fit_resample(X_train, y_train)

# Step 2: Apply SMOTE (Oversampling - Generate Minority Samples)
smote = SMOTE(sampling_strategy=1, random_state=1)
X_train_final, y_train_final = smote.fit_resample(X_train_enn, y_train_enn)

# Print new class distribution
print(f"After ENN + SMOTE, counts of label '1': {sum(y_train_final == 1)}")
print(f"After ENN + SMOTE, counts of label '0': {sum(y_train_final == 0)} \n")

# Print final dataset shape
print(f"After ENN + SMOTE, shape of train_X: {X_train_final.shape}")
print(f"After ENN + SMOTE, shape of train_y: {y_train_final.shape} \n")


In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train_final, y=y_train_final, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train_final, y_train_final)
    scores = recall_score(y_test, model.predict(X_test))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

In [ ]:
from imblearn.under_sampling import EditedNearestNeighbours

# Apply EditedNearestNeighbours (ENN) to remove misclassified majority class samples
enn = EditedNearestNeighbours()
X_train_final, y_train_final = enn.fit_resample(X_train, y_train)

# Print class distribution after ENN
print(f"After ENN, counts of label '1': {sum(y_train_final == 1)}")
print(f"After ENN, counts of label '0': {sum(y_train_final == 0)} \n")

# Print dataset shape
print(f"After ENN, shape of train_X: {X_train_final.shape}")
print(f"After ENN, shape of train_y: {y_train_final.shape} \n")

# Features selection

In [ ]:
%%time

import optuna.logging

# Set the logging level to INFO to display optimization progress
optuna.logging.set_verbosity(optuna.logging.INFO)

# Define the objective function to optimize
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 30, 500)
    max_depth = trial.suggest_int('max_depth', 10, 500, step=10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    # Create the ExtraTrees classifier using suggested hyperparameters
    clf = ExtraTreesClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        random_state=42
    )

    # Repeated K-Fold cross-validator
    rkf = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)

    # Perform cross-validation
    scores = cross_val_score(clf, X_train_final, y_train_final, cv=rkf, scoring='recall')

    # Return the mean F1 score
    return np.mean(scores)

# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)  # Adjust the number of trials as necessary

# Print the best parameters
print('Best trial:', study.best_trial.params)

# Fit the model using the best parameters found
best_model = ExtraTreesClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train_final, y_train_final)

# Print feature importances
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]
print("Feature ranking:")
for f in range(X_train_final.shape[1]):
    print(f"{f + 1}. feature {X_train_final.columns[indices[f]]} ({importances[indices[f]]})")

# Optional: Plot graph of feature importances for better visualization
feat_importances = pd.Series(best_model.feature_importances_, index=X_train_final.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title('Top 10 Important Features from Tuned Model')
plt.xlabel('Relative Importance')
plt.show()

In [ ]:
# Assuming `best_model` is your trained model
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Select the number of features you want to plot
n_features_to_plot = 10

# Create a Series with feature importances and names
feat_importances = pd.Series(importances[indices[:n_features_to_plot]], index=X_train_final.columns[indices[:n_features_to_plot]])

# Plot
plt.figure(figsize=(10, 8))
feat_importances.plot(kind='barh')
plt.title('Top 20 Important Features')
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()  # Invert y-axis to have the most important at the top
plt.show()

In [ ]:
# Calculate cumulative importance
cumulative_importances = np.cumsum(np.sort(importances)[::-1])

# Find the number of features needed for 90% of total importance
n_90_percent = np.where(cumulative_importances > 0.90)[0][0] + 1

# Plot these features
feat_importances = pd.Series(importances[indices[:n_90_percent]], index=X_train_final.columns[indices[:n_90_percent]])
plt.figure(figsize=(10, 8))
feat_importances.plot(kind='barh')
plt.title(f'Top {n_90_percent} Features to Reach 90% Cumulative Importance')
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()  # Invert y-axis to have the most important at the top
plt.show()


In [ ]:
# Get the sorted feature importances
sorted_importances = sorted(list(zip(X_train_final.columns, best_model.feature_importances_)), key=lambda x: x[1], reverse=True)

# Initialize cumulative importance and feature list
cumulative_importance = 0.0
important_features = []

# Loop over sorted importances
for feature, importance in sorted_importances:
    cumulative_importance += importance
    important_features.append(feature)
    if cumulative_importance >= 0.90:
        break

print("Features reaching 90% cumulative importance:")
for feature in important_features:
    print(feature)

In [ ]:
X_train1=X_train_final[important_features]
X_test1=X_test[important_features]
print(f"X_train1 shape: {X_train1.shape}, X_test1 shape: {X_test1.shape}")

In [ ]:
X_train1.head()

In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))     
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1,verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models

# loop through all models to get the mean cross validated score
print("\nCross-Validation performance on training dataset:\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(n_splits=10,n_repeats=5, random_state=1)  # Setting number of splits equal to 5
    cv_result = cross_val_score(estimator=model, X=X_train1, y=y_train_final, scoring=scorer, cv=kfold)
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\nValidation Performance:\n")

for name, model in models:
    model.fit(X_train1, y_train_final)
    scores = recall_score(y_test, model.predict(X_test1))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=45)

plt.show()

### Principal Component Analysis

Principal Component Analysis, or PCA for short, is a method for reducing the dimensionality(number of features) of data.

It can be thought of as a projection method where data with m-columns (features) is projected into a subspace with m or fewer columns, whilst retaining the essence of the original data.

In [ ]:
X_train1.shape

In [ ]:
covMatrix = np.cov(X_train1,rowvar=False)
print(covMatrix)

In [ ]:
pca = PCA(n_components=17)
pca.fit(X_train1)

In [ ]:
plt.bar(list(range(1,18)),pca.explained_variance_ratio_,alpha=0.5, align='center')
plt.ylabel('Variation explained')
plt.xlabel('eigen Value')
plt.show()

In [ ]:
plt.step(list(range(1,18)),np.cumsum(pca.explained_variance_ratio_), where='mid')
plt.ylabel('Cum of variation explained')
plt.xlabel('eigen Value')
plt.show()

In [ ]:
pca10 = PCA(n_components=6)
pca10.fit(X_train1)
print(pca10.components_)
print(pca10.explained_variance_ratio_)
X_trainpca10 = pca10.transform(X_train1)

In [ ]:
pca10 = PCA(n_components=6)
pca10.fit(X_test1)
print(pca10.components_)
print(pca10.explained_variance_ratio_)
X_testpca10 = pca10.transform(X_test1)

In [ ]:
X_trainpca10.shape,X_testpca10.shape

In [ ]:
%%time

models = []  # Empty list to store all the models

# Appending models into the list

models.append(("dtree", DecisionTreeClassifier(random_state=1)))
models.append(("Lr", LogisticRegression(random_state=1)))
models.append(("KNeighbors",KNeighborsClassifier()))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))
models.append(("LightGBM", LGBMClassifier(random_state=1, verbose=-1)))
models.append(("Catboost", CatBoostClassifier(random_state=1,verbose=False)))


results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models


# loop through all models to get the mean cross validated score
print("\n" "Cross-Validation performance on training dataset:" "\n")

for name, model in models:
    kfold = RepeatedStratifiedKFold(
        n_splits=10,n_repeats=5, random_state=1
    )  # Setting number of splits equal to 5
    cv_result = cross_val_score(
        estimator=model, X=X_trainpca10, y=y_train, scoring=scorer, cv=kfold
    )
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_trainpca10, y_train)
    scores = f1_score(y_test, model.predict(X_testpca10))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(15, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)
plt.xticks(rotation=90)

plt.show()

## HyperparameterTuning

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve

# Define function to find the best threshold for F1-score while ensuring Recall > Precision
def tune_threshold_f1_with_recall(y_true, y_pred_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)  # Get thresholds
    
    best_f1 = 0
    best_threshold = 0

    for t in thresholds:
        y_pred = (y_pred_proba >= t).astype(int)
        recall = recall_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        
        # Condition: Ensure Recall > Precision while maximizing F1-score
        if recall > precision and f1 > best_f1:
            best_f1 = f1
            best_threshold = t
    
    return best_threshold


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, roc_curve

# Define function to find the best threshold for maximum recall while ensuring Precision >= min_precision
def tune_threshold_recall_focus(y_true, y_pred_proba, min_precision=0.5):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    
    best_recall = 0
    best_threshold = 0

    for t in thresholds:
        y_pred = (y_pred_proba >= t).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred)

        # Condition: Ensure precision is acceptable while maximizing recall
        if precision >= min_precision and recall > best_recall:
            best_recall = recall
            best_threshold = t

    return best_threshold


### Catboost 

In [ ]:
X_train1.shape

In [ ]:
%%time

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.integration import CatBoostPruningCallback


# ✅ Define the safe objective function generator
def make_objective(X_train1, y_train_final, X_test1, y_test):
    def objective(trial):
        param_space = {
            "iterations": trial.suggest_int("iterations", 500, 700),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "depth": trial.suggest_int("depth", 3, 13),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 7.0, log=True),
            "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian"]),
            "random_strength": trial.suggest_float("random_strength", 1e-8, 1e-2, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.2, 0.8),
            "od_type": trial.suggest_categorical("od_type", ["Iter"]),
            "od_wait": trial.suggest_int("od_wait", 17, 45),
            "border_count": trial.suggest_int("border_count", 100, 160),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.2, 1.5)
        }

        model = CatBoostClassifier(**param_space, verbose=False)
        model.fit(X_train1, y_train_final,
                  eval_set=[(X_test1, y_test)],
                  early_stopping_rounds=100,
                  callbacks=[CatBoostPruningCallback(trial, "Logloss")])

        y_pred = model.predict(X_test1)
        return recall_score(y_test, y_pred)

    return objective

# ✅ Create your study
optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler = TPESampler(seed=1)
pruner = MedianPruner(n_warmup_steps=50)
study1 = optuna.create_study(study_name="catboost_recall", direction="maximize", sampler=sampler, pruner=pruner)

# ✅ Replace these with your actual data
# Xtrain, ytrain, Xtest, ytest should already be defined
# Now run optimization
objective_fn = make_objective(X_train1, y_train_final, X_test1, y_test)
study1.optimize(objective_fn, n_trials=100)

# ✅ Print the results
print("Number of finished trials:", len(study1.trials))
print("Best trial:")
trial = study1.best_trial
print("  Recall:", trial.value)
print("  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# ✅ Train the final model using the best parameters
catboost_tuned = CatBoostClassifier(**trial.params, verbose=False)
catboost_tuned.fit(X_train1, y_train_final)


In [ ]:
%%time

# Define your CatBoost model (or replace with your model)
cat_model = CatBoostClassifier()

# Define a smaller Grid Search space around Optuna's best parameters
param_grid = {
    "iterations": [591], 
    "learning_rate": [0.3], 
    "depth": [5,8,15],
    "l2_leaf_reg": [5.53],
    "bootstrap_type": ["Bayesian"],
    "random_strength": [0.0004],
    "bagging_temperature": [0.32],
    "od_type": ["Iter"],
    "od_wait": [35],
    "border_count": [130],
    "scale_pos_weight": [1.38]
}

# Define K-Fold cross-validation strategy
kfold = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=1)

# Perform Grid Search
grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    scoring="f1",
    cv=kfold,
    n_jobs=-1
)

# Fit the model on undersampled data (assuming X_train1 and y_train are defined)
grid_search.fit(X_train1, y_train_final)

# Print best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)


In [ ]:
# Define the tuned CatBoost model with corrected parameter syntax
catboost_tuned = CatBoostClassifier(
    iterations=557, 
    learning_rate=0.095, 
    depth=8,
    l2_leaf_reg=3.2904238003974,
    bootstrap_type="Bayesian",
    random_strength=7.248881590936032e-06,
    bagging_temperature=0.7177056070370484,
    od_type="Iter",
    od_wait=43,
    border_count=127,
    scale_pos_weight=1.4085535107956455,
    verbose=100  # Optional: Controls logging frequency
)

# Fit the model on the training data
catboost_tuned.fit(X_train1, y_train_final)

In [ ]:
# Generate predicted probabilities
y_pred_proba = catboost_tuned.predict_proba(X_test1)[:, 1]

# Use the function
best_threshold = tune_threshold_recall_focus(y_test, y_pred_proba)
print("Best Threshold for F1-score (with Recall > Precision):", best_threshold)


In [ ]:
# Generate predicted probabilities
y_pred_proba = catboost_tuned.predict_proba(X_test1)[:, 1]

# Use the recall-focused threshold tuning function
best_threshold = tune_threshold_recall_focus(y_test, y_pred_proba, min_precision=0.6)

print("Best Threshold for Recall (with Precision ≥ 0.6):", best_threshold)


In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, roc_curve

# Get probability predictions (assuming lightgbm or catboost model)
y_probs = catboost_tuned.predict_proba(X_test1)[:, 1]  

# Define a range of thresholds
thresholds = np.linspace(0.01, 0.9, 20)  # Try different values

recalls = []
precisions = []
f1_scores = []

# Evaluate model at different thresholds
for threshold in thresholds:
    y_pred_adjusted = (y_probs >= threshold).astype(int)
    
    recalls.append(recall_score(y_test, y_pred_adjusted))
    precisions.append(precision_score(y_test, y_pred_adjusted))
    f1_scores.append(f1_score(y_test, y_pred_adjusted))

# Plot Recall vs. Threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, label="Recall", marker="o")
plt.plot(thresholds, precisions, label="Precision", marker="s")
plt.plot(thresholds, f1_scores, label="F1 Score", marker="^")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning: Recall vs Precision vs F1")
plt.legend()
plt.grid()
plt.show()

# Choose the best threshold (e.g., highest recall with reasonable precision)
best_threshold = thresholds[np.argmax(recalls)]
print(f"Best Threshold for Recall: {best_threshold:.2f}")

# Make final predictions with the best threshold
y_pred_final = (y_probs >= best_threshold).astype(int)

# Check final recall
final_recall = recall_score(y_test, y_pred_final)
print(f"Final Recall: {final_recall:.4f}")

In [ ]:
# Calculating different metrics on train set
catboost_random_train = model_performance_classification_sklearn(
    catboost_tuned, X_train1, y_train_final,threshold= 0.5
)
print("training performance:")
catboost_random_train

In [ ]:
# Calculating different metrics on validation set
catboost_random_test = model_performance_classification_sklearn(catboost_tuned, X_test1, y_test,threshold= 0.15)
print("Validation performance:")
catboost_random_test

In [ ]:
# creating confusion matrix
confusion_matrix_sklearn(catboost_tuned, X_test1, y_test,threshold= 0.15)

### LGBMboost 

In [ ]:
%%time


import lightgbm as lgb

# Define the objective function for Optuna
def objective(trial):
    param = {
        'learning_rate': trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
        'num_leaves': trial.suggest_int("num_leaves", 2, 256),
        'feature_fraction': trial.suggest_float("feature_fraction", 0.1, 1.0),
        'bagging_fraction': trial.suggest_float("bagging_fraction", 0.1, 1.0),
        'bagging_freq': trial.suggest_int("bagging_freq", 1, 7),
        'max_depth': trial.suggest_int("max_depth", -1, 50),
        'verbosity': -1,
        'lambda_l1': trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float("min_split_gain", 0.0, 1.0),
        'zero_as_missing': False,
        'max_bin': trial.suggest_int("max_bin", 255, 500),
        'min_data_in_bin': trial.suggest_int("min_data_in_bin", 1, 10),
        'random_state': 42,
        'device_type': "cpu",
        'objective': "binary",
        'metric': "binary_logloss",
        'min_sum_hessian_in_leaf': trial.suggest_float("min_sum_hessian_in_leaf", 1e-3, 10.0, log=True),
        'num_iterations': trial.suggest_int("num_iterations", 100, 1000),
    }
    gbm = lgb.LGBMClassifier(**param)
    gbm.fit(X_train1, y_train_final)
    preds = gbm.predict(X_test1)
    recall = recall_score(y_test, preds)
    return recall

# Create the Optuna study and optimize
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# Print the best parameters and the best F1 score
print('Best parameters:', study.best_params)
print('Best value:', study.best_value)
print('Best trial:', study.best_trial)

# Train the final model using the best parameters
lightgbm_tuned = lgb.LGBMClassifier(**study.best_params)
lightgbm_tuned.fit(X_train1, y_train_final)

In [ ]:
%%time

# Define a smaller Grid Search space around Optuna’s best parameters for LGBMClassifier
param_grid = {
    "num_iterations": [10000],  
    "learning_rate": [0.009],  
    "max_depth": [24],  
    "num_leaves": [16],  
    "max_bin": [330],  
    "min_data_in_bin": [8], 
    "feature_fraction": [0.82],  
    "bagging_fraction": [0.48],  
    "bagging_freq": [6],  
    "lambda_l1": [3.2679722523601793],  
    "lambda_l2": [0.013308803442221845], 
    "min_split_gain": [0.27], 
    "min_sum_hessian_in_leaf": [1.6] 
}

# Define K-Fold cross-validation strategy
kfold = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=1)

# Initialize LGBMClassifier
lgbm_model = LGBMClassifier()

# Perform Grid Search
grid_search = GridSearchCV(
    estimator=lgbm_model,
    param_grid=param_grid,
    scoring="recall",
    cv=kfold,
    n_jobs=-1
)

# Fit the model on undersampled data (assuming X_train1 and y_train are defined)
grid_search.fit(X_train1, y_train_final)

# Print best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)


In [ ]:
# Print best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

In [ ]:
%%time

import lightgbm as lgb
# Define the tuned LightGBM model with corrected parameter syntax
lightgbm_tuned = lgb.LGBMClassifier(
    num_iterations=198,  
    learning_rate=0.0015647803379510246,  
    max_depth=18,  
    num_leaves=5,  
    max_bin=349, 
    min_data_in_bin=8, 
    feature_fraction=0.6183441390545854,  
    bagging_fraction=0.3211865788702094,  
    bagging_freq=6,  
    lambda_l1=3.2679722523601793,  
    lambda_l2=0.013308803442221845, 
    min_split_gain=0.5730223674745507, 
    min_sum_hessian_in_leaf=2.5078613492776127
)

# Fit the model on the training data
lightgbm_tuned.fit(X_train1, y_train_final)


In [ ]:
# Get probability predictions (assuming lightgbm or catboost model)
y_probs = lightgbm_tuned.predict_proba(X_test1)[:, 1]  

# Define a range of thresholds
thresholds = np.linspace(0.01, 0.9, 20)  # Try different values

recalls = []
precisions = []
f1_scores = []

# Evaluate model at different thresholds
for threshold in thresholds:
    y_pred_adjusted = (y_probs >= threshold).astype(int)
    
    recalls.append(recall_score(y_test, y_pred_adjusted))
    precisions.append(precision_score(y_test, y_pred_adjusted))
    f1_scores.append(f1_score(y_test, y_pred_adjusted))

# Plot Recall vs. Threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, label="Recall", marker="o")
plt.plot(thresholds, precisions, label="Precision", marker="s")
plt.plot(thresholds, f1_scores, label="F1 Score", marker="^")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning: Recall vs Precision vs F1")
plt.legend()
plt.grid()
plt.show()

# Choose the best threshold (e.g., highest recall with reasonable precision)
best_threshold = thresholds[np.argmax(recalls)]
print(f"Best Threshold for Recall: {best_threshold:.2f}")

# Make final predictions with the best threshold
y_pred_final = (y_probs >= best_threshold).astype(int)

# Check final recall
final_recall = recall_score(y_test, y_pred_final)
print(f"Final Recall: {final_recall:.4f}")

In [ ]:
# Calculating different metrics on train set
lightgbm_random_train = model_performance_classification_sklearn(
    lightgbm_tuned, X_train1, y_train_final,threshold= 0.5
)
print("Training performance:")
lightgbm_random_train

In [ ]:
# Calculating different metrics on validation set
lightgbm_random_test = model_performance_classification_sklearn(lightgbm_tuned, X_test1, y_test,threshold= 0.5)
print("testing performance:")
lightgbm_random_test

In [ ]:
# creating confusion matrix
confusion_matrix_sklearn(lightgbm_tuned, X_test1, y_test,threshold= 0.5)

### XGboost

In [ ]:
%%time

# Define the objective function for Optuna
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 25, 55),  
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.18, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 390, 490),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 6), 
        'gamma': trial.suggest_loguniform('gamma', 0.1, 1),
        'subsample': trial.suggest_uniform('subsample', 0.70, 0.95),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-7, 1e-5),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 1e-6),
        'scale_pos_weight': trial.suggest_uniform('scale_pos_weight', 1.1, 1.5),
        'objective': 'binary:logistic',
        'eval_metric': 'logloss'  # ✅ Ensure evaluation metric is defined
    }

    optuna_model = XGBClassifier(**params)

    # ✅ Remove 'early_stopping_rounds' to avoid errors
    optuna_model.fit(
        X_train1, y_train_final,
        eval_set=[(X_test1, y_test)], 
        verbose=False
    )

    # Get predictions and compute F1-score
    y_pred = optuna_model.predict(X_test1)
    recall = recall_score(y_test, y_pred)
    return recall

# Set Optuna verbosity to reduce excessive logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Initialize the sampler and pruner
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=30, interval_steps=10)
sampler = TPESampler(seed=42)

# Create the study and optimize
study8 = optuna.create_study(direction='maximize', pruner=pruner, sampler=sampler)
study8.optimize(objective, n_trials=100, n_jobs=-1)

# Display the results of the optimization
print('Number of finished trials: {}'.format(len(study8.trials)))
print('Best trial:')
trial = study8.best_trial

print('  Value: {}'.format(trial.value))
print('  Params: ')
for key, value in trial.params.items():
    print('    {}: {}'.format(key, value))

# Train the final model using the best parameters
params = trial.params
XGB_tuned = XGBClassifier(**params)
XGB_tuned.fit(X_train1, y_train_final)  


In [ ]:
# Get probability predictions (assuming lightgbm or catboost model)
y_probs = XGB_tuned.predict_proba(X_test1)[:, 1]  

# Define a range of thresholds
thresholds = np.linspace(0.01, 0.9, 20)  # Try different values

recalls = []
precisions = []
f1_scores = []

# Evaluate model at different thresholds
for threshold in thresholds:
    y_pred_adjusted = (y_probs >= threshold).astype(int)
    
    recalls.append(recall_score(y_test, y_pred_adjusted))
    precisions.append(precision_score(y_test, y_pred_adjusted))
    f1_scores.append(f1_score(y_test, y_pred_adjusted))

# Plot Recall vs. Threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, label="Recall", marker="o")
plt.plot(thresholds, precisions, label="Precision", marker="s")
plt.plot(thresholds, f1_scores, label="F1 Score", marker="^")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning: Recall vs Precision vs F1")
plt.legend()
plt.grid()
plt.show()

# Choose the best threshold (e.g., highest recall with reasonable precision)
best_threshold = thresholds[np.argmax(recalls)]
print(f"Best Threshold for Recall: {best_threshold:.2f}")

# Make final predictions with the best threshold
y_pred_final = (y_probs >= best_threshold).astype(int)

# Check final recall
final_recall = recall_score(y_test, y_pred_final)
print(f"Final Recall: {final_recall:.4f}")

In [ ]:
# Calculating different metrics on train set
XGB_random_train = model_performance_classification_sklearn(
    XGB_tuned, X_train1, y_train_final,threshold= 0.5
)
print("Training performance:")
XGB_random_train

In [ ]:
# Calculating different metrics on validation set
XGB_random_test = model_performance_classification_sklearn(XGB_tuned, X_test1, y_test,threshold= 0.2)
print("testing performance:")
XGB_random_test

In [ ]:
# creating confusion matrix
confusion_matrix_sklearn(XGB_tuned, X_test1, y_test,threshold= 0.3)

### Random Forest

In [ ]:
%%time


# Define the objective function for Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),  # Number of trees
        'max_depth': trial.suggest_int('max_depth', 10, 100),  # Depth of trees
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),  # Min samples to split
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),  # Min samples per leaf
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),  # Features used per split
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),  # Bootstrap sampling
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),  # Handle imbalanced data
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss']),  # Splitting criterion
        'random_state': 42  # Keep consistency
    }

    rf_model = RandomForestClassifier(**params)

    rf_model.fit(X_train1, y_train_final)  # Train on dataset

    # Get predictions and compute F1-score
    y_pred = rf_model.predict(X_test1)
    recall = recall_score(y_test, y_pred)
    
    return recall

# Set Optuna verbosity to reduce excessive logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Initialize the sampler and pruner
pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=30, interval_steps=10)
sampler = TPESampler(seed=42)

# Create the study and optimize
study_rf = optuna.create_study(direction='maximize', pruner=pruner, sampler=sampler)
study_rf.optimize(objective, n_trials=100, n_jobs=-1)

# Display the results of the optimization
print('Number of finished trials: {}'.format(len(study_rf.trials)))
print('Best trial:')
best_trial = study_rf.best_trial

print('  Value: {}'.format(best_trial.value))
print('  Params: ')
for key, value in best_trial.params.items():
    print('    {}: {}'.format(key, value))

# Train the final model using the best parameters
best_params = best_trial.params
RF_tuned = RandomForestClassifier(**best_params)
RF_tuned.fit(X_train1, y_train_final)  # Train the final model

In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

# Define a smaller Grid Search space around Optuna’s best RandomForest parameters
param_grid = {
    "n_estimators": [630],   
    "max_depth": [52],              
    "min_samples_split": [2],            
    "min_samples_leaf": [1],              
    "max_features": ["sqrt"],             
    "bootstrap": [True],                 
    "class_weight": ["balanced_subsample"],         
    "criterion": ["gini"]   
}

# Define K-Fold cross-validation strategy
kfold = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=1)

# Initialize RandomForestClassifier
rf_model = RandomForestClassifier(random_state=1)

# Perform Grid Search
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring="f1",
    cv=kfold,
    n_jobs=-1
)

# Fit the model on training data (assuming X_train1 and y_train are defined)
grid_search.fit(X_train1, y_train_final)

# Print best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)


In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier

# Define the tuned RandomForest model with your best parameters
RF_tuned = RandomForestClassifier(
    n_estimators=638,
    max_depth=10,
    min_samples_split=7,
    min_samples_leaf=3,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced_subsample',
    criterion='gini'
)

# Fit the model on the training data
RF_tuned.fit(X_train1, y_train_final)


In [ ]:
# Get probability predictions (assuming lightgbm or catboost model)
y_probs = RF_tuned.predict_proba(X_test1)[:, 1]  

# Define a range of thresholds
thresholds = np.linspace(0.01, 0.9, 30)  # Try different values

recalls = []
precisions = []
f1_scores = []

# Evaluate model at different thresholds
for threshold in thresholds:
    y_pred_adjusted = (y_probs >= threshold).astype(int)
    
    recalls.append(recall_score(y_test, y_pred_adjusted))
    precisions.append(precision_score(y_test, y_pred_adjusted))
    f1_scores.append(f1_score(y_test, y_pred_adjusted))

# Plot Recall vs. Threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, label="Recall", marker="o")
plt.plot(thresholds, precisions, label="Precision", marker="s")
plt.plot(thresholds, f1_scores, label="F1 Score", marker="^")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning: Recall vs Precision vs F1")
plt.legend()
plt.grid()
plt.show()

# Choose the best threshold (e.g., highest recall with reasonable precision)
best_threshold = thresholds[np.argmax(recalls)]
print(f"Best Threshold for Recall: {best_threshold:.2f}") 

# Make final predictions with the best threshold
y_pred_final = (y_probs >= best_threshold).astype(int)

# Check final recall
final_recall = recall_score(y_test, y_pred_final)
print(f"Final Recall: {final_recall:.4f}")

In [ ]:
# Calculating different metrics on train set
RF_random_train = model_performance_classification_sklearn(
    RF_tuned, X_train1, y_train_final,threshold= 0.5
)
print("Training performance:")
RF_random_train

In [ ]:
# Calculating different metrics on validation set
RF_random_test = model_performance_classification_sklearn(RF_tuned, X_test1, y_test,threshold= 0.41)
print("testing performance:")
RF_random_test

In [ ]:
# Custom thresholds
thresholds = {
    'Random Forest Tuned': 0.41,
    'XGBoost Tuned': 0.04,
    'LightGBM Tuned': 0.5,
    'CatBoost Tuned': 0.08,
}

In [ ]:
# creating confusion matrix
confusion_matrix_sklearn(RF_tuned, X_test1, y_test,threshold= 0.41)

In [ ]:
# training performance comparison

models_test_comp_df = pd.concat(
    [ catboost_random_train.T,
      lightgbm_random_train.T,
      XGB_random_train.T,
     RF_random_train.T,
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Catboost tuned with ENN data",
    "LGBMboost tuned with ENN data",
    "XGboost tuned with ENN data",
    'RandomForest tuned with ENN data',
]
print("Training performance comparison:")
models_test_comp_df

In [ ]:
# validation performance comparison

models_val_comp_df = pd.concat(
    [ catboost_random_test.T,
      lightgbm_random_test.T,
      XGB_random_test.T,
     RF_random_test.T,
    ],
    axis=1,
)
models_val_comp_df.columns = [
    "Catboost tuned with ENN data",
    "LGBMboost tuned with ENN data",
    "XGboost tuned with ENN data",
    'RandomForest tuned with ENN data',
]
print("validation performance comparison:")
models_val_comp_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your models
models = {
    'Random Forest Tuned': RF_tuned,
    'XGBoost Tuned': XGB_tuned,
    'LightGBM Tuned': lightgbm_tuned,
    'CatBoost Tuned': catboost_tuned,
}

# Custom thresholds
thresholds = {
    'Random Forest Tuned': 0.41,
    'XGBoost Tuned': 0.3,
    'LightGBM Tuned': 0.5,
    'CatBoost Tuned': 0.15,
}

results = {'training': {}, 'test': {}}

# Loop over models and compute performance with custom thresholds
for name, model in models.items():
    thr = thresholds.get(name, 0.5)

    # TRAIN
    train_perf = model_performance_classification_sklearn(model, X_train1, y_train_final, threshold=thr)
    results['training'][name] = train_perf["Recall"].values[0]

    # TEST
    test_perf = model_performance_classification_sklearn(model, X_test1, y_test, threshold=thr)
    results['test'][name] = test_perf["Recall"].values[0]

# ✅ Plotting
model_names = list(results['test'].keys())

fig, ax = plt.subplots(figsize=(12, 7))
bar_width = 0.35
index = np.arange(len(model_names))

for i, dataset in enumerate(['training', 'test']):
    scores = [results[dataset][m] for m in model_names]
    bar_positions = index + (i * bar_width)
    bars = ax.bar(bar_positions, scores, bar_width, label=dataset.capitalize())

    # Add % labels
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f'{height:.2%}',
            ha='center',
            va='bottom'
        )

ax.set_title('Model Comparison — Recall (custom thresholds)')
ax.set_xlabel('Models')
ax.set_ylabel('Recall')
ax.set_xticks(index + bar_width / 2)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Assuming the rest of your code and model definitions remain the same

models = {
    'Random forest tuned': RF_tuned,
    'XGBoost tuned': XGB_tuned,
    'LightGBM tuned': lightgbm_tuned,
    'CatBoost tuned': catboost_tuned
}

plt.figure(figsize=(8, 6))

# Ensure y_test is correctly defined before this point
print("y_test type:", type(y_test))
try:
    print("y_test shape:", y_test.shape)
except AttributeError:
    print("y_test does not have a shape attribute, indicating it might not be correctly defined.")

for model_name, model in models.items():
    probs = model.predict_proba(X_test1)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Uvl Prediction Models')
plt.legend(loc='lower right')
plt.grid()
plt.show()

In [ ]:
X_train1.shape

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1) Extract importances from LightGBM model
feature_importances = lightgbm_tuned.feature_importances_

# 2) Get feature names (fall back if needed)
try:
    feature_names = list(X_train1.columns)
except Exception:
    feature_names = [f"feature_{i}" for i in range(len(feature_importances))]

# 3) Build sorted df and normalize to %
importance_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
      .sort_values('Importance', ascending=False)
)
importance_df['Importance'] = (
    importance_df['Importance'] / importance_df['Importance'].sum() * 100
)

# 4) Plot (shows top 30 for readability)
top_n = 30
plot_df = importance_df.head(top_n)

plt.figure(figsize=(10, 6))
plt.barh(plot_df['Feature'], plot_df['Importance'])
plt.xlabel('Feature Importance (%)')
plt.ylabel('Features')
plt.title('Feature Importance — Tuned LightGBM (Top 30)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

# -------- choose contributing features --------
def contributing_used_lgbm(model, X):
    split = model.booster_.feature_importance(importance_type='split')
    gain  = model.booster_.feature_importance(importance_type='gain')
    names = np.array(X.columns)
    mask  = (split > 0) | (gain > 0)
    return list(names[mask])

def contributing_permutation(model, X, y, scoring="recall", n_repeats=10, random_state=42):
    names = np.array(X.columns)
    perm = permutation_importance(model, X, y, scoring=scoring,
                                  n_repeats=n_repeats, random_state=random_state, n_jobs=-1)
    # build a sorted dataframe for transparency
    df = pd.DataFrame({
        "Feature": names,
        "PI_mean": perm.importances_mean,
        "PI_std":  perm.importances_std
    }).sort_values("PI_mean", ascending=False)
    # keep strictly positive mean importance
    keep = df[df["PI_mean"] > 0]["Feature"].tolist()
    return keep, df

# 1) features used by LightGBM
used_feats = contributing_used_lgbm(lightgbm_tuned, X_train1)

# 2) features that matter for RECALL on the TEST set
recall_feats, perm_table = contributing_permutation(lightgbm_tuned, X_test1, y_test, scoring="recall")

# >>> FINAL shortlist for LR (recall-focused)
features_for_lr = recall_feats          # to be used for direction/magnitude with LR

print(f"{len(used_feats)} features 'used' by trees:\n", used_feats)
print(f"\n{len(recall_feats)} features that improve recall (permutation):\n", features_for_lr)

# (Optional) If you want a broader set, use the union:
# features_for_lr = sorted(set(used_feats) | set(recall_feats))

# (Optional) Save the ordered permutation table for your report
# perm_table.to_csv("permutation_importance_recall.csv", index=False)


In [ ]:
features_for_lr = sorted(set(used_feats) | set(recall_feats))
features_for_lr

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# --- 1) features used by LightGBM (any split/gain > 0) ---
def lgbm_used_features(model, X):
    split = model.booster_.feature_importance(importance_type='split')
    gain  = model.booster_.feature_importance(importance_type='gain')
    names = np.array(X.columns)
    mask  = (split > 0) | (gain > 0)
    return list(names[mask])

# --- 2) permutation-importance features for RECALL on test set ---
def perm_features_for_recall(model, X, y, n_repeats=10, random_state=42):
    names = np.array(X.columns)
    perm = permutation_importance(
        model, X, y, scoring="recall",
        n_repeats=n_repeats, random_state=random_state, n_jobs=-1
    )
    perm_df = pd.DataFrame({
        "Feature": names,
        "PI_mean": perm.importances_mean,
        "PI_std":  perm.importances_std
    }).sort_values("PI_mean", ascending=False)
    keep = perm_df.loc[perm_df["PI_mean"] > 0, "Feature"].tolist()
    return keep, perm_df

# --- compute sets ---
used_feats = lgbm_used_features(lightgbm_tuned, X_train1)
recall_feats, perm_table = perm_features_for_recall(lightgbm_tuned, X_test1, y_test)

# --- union for LR ---
features_for_lr = sorted(set(used_feats) | set(recall_feats))
print(f"{len(features_for_lr)} features_for_lr:\n", features_for_lr)

# --- build plotting frame (keep only union rows from permutation table; fill 0 if not present) ---
pi = perm_table.set_index("Feature")["PI_mean"]
plot_df = pd.DataFrame({"Feature": features_for_lr})
plot_df["PI_mean"] = plot_df["Feature"].map(pi).fillna(0.0)

# Tag membership: Both / Permutation only / Used only
plot_df["Membership"] = np.where(
    plot_df["Feature"].isin(recall_feats) & plot_df["Feature"].isin(used_feats), "Both",
    np.where(plot_df["Feature"].isin(recall_feats), "Permutation only", "Used only")
)

# Sort by PI_mean (then by Membership to keep "Both" near top if ties)
plot_df = plot_df.sort_values(["PI_mean", "Membership"], ascending=[True, True])

# --- plot ---
fig, ax = plt.subplots(figsize=(12, 7))
for label in ["Both", "Permutation only", "Used only"]:
    sub = plot_df[plot_df["Membership"] == label]
    if not sub.empty:
        ax.barh(sub["Feature"], sub["PI_mean"], label=label)

ax.set_xlabel("Permutation importance (recall)")
ax.set_ylabel("Feature")
ax.set_title("Features selected (union of Used & Permutation-important)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
X_train= X_train1[features_for_lr]
X_train.shape

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Ensure DataFrame + numeric dtype
Xvif = pd.DataFrame(X_train).copy()
Xvif = Xvif.apply(pd.to_numeric, errors='coerce')           # force numeric
Xvif = Xvif.replace([np.inf, -np.inf], np.nan)              # remove infs
# If any columns became all-NaN, drop them
Xvif = Xvif.dropna(axis=1, how='all')

# Drop constant columns (VIF undefined / infinite)
const_cols = [c for c in Xvif.columns if Xvif[c].nunique(dropna=True) <= 1]
if const_cols:
    Xvif = Xvif.drop(columns=const_cols)

# Drop any remaining rows with NaN (should be none after your pipeline)
Xvif = Xvif.dropna(axis=0)

# Cast to float64 (fixes the TypeError)
Xvif = Xvif.astype(np.float64)

# (Recommended) add intercept for VIF calc, then skip its VIF
X_with_const = sm.add_constant(Xvif, has_constant='add')

vif_vals = []
vif_names = Xvif.columns.tolist()
for i in range(1, X_with_const.shape[1]):  # skip the intercept at index 0
    vif_vals.append(variance_inflation_factor(X_with_const.values, i))

vif_series = pd.Series(vif_vals, index=vif_names, name='VIF').sort_values(ascending=False)

print("Dropped constant columns:", const_cols)
print(vif_series.head(20))


In [ ]:
%%time

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def _compute_vif(df_num: pd.DataFrame) -> pd.DataFrame:
    """Compute VIFs on a numeric DataFrame; imputes NaNs with medians for VIF math."""
    X = df_num.copy()
    med = X.median(numeric_only=True)
    X = X.fillna(med)

    Xc = sm.add_constant(X, has_constant='add')
    vifs = []
    for i, col in enumerate(Xc.columns):
        if col == 'const':
            continue
        try:
            v = float(variance_inflation_factor(Xc.values, i))
        except Exception:
            v = np.inf
        vifs.append((col, v))
    return pd.DataFrame(vifs, columns=['Feature', 'VIF']).sort_values('VIF', ascending=False).reset_index(drop=True)

def reduce_vif(X: pd.DataFrame,
               thresh: float = 10.0,
               keep: list[str] | None = None,
               max_iter: int = 100):
    """
    Iteratively drop the highest-VIF column until all VIF <= thresh.
    'keep' columns (if any) are never dropped (even if VIF > thresh).
    Returns: X_reduced, removed_zero_var, removed_vif, final_vif_df
    """
    keep = keep or []
    X = X.copy()

    # Restrict to numeric for VIF math
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric = [c for c in X.columns if c not in numeric_cols]
    if non_numeric:
        print(f"⚠️ Ignoring non-numeric columns for VIF computation: {non_numeric}")

    X_num = X[numeric_cols].copy()

    # Drop zero-variance columns (cannot compute meaningful VIF)
    zero_var = [c for c in X_num.columns if X_num[c].nunique(dropna=False) <= 1]
    if zero_var:
        print(f"⚠️ Dropping zero-variance columns: {zero_var}")
        X_num.drop(columns=zero_var, inplace=True, errors='ignore')

    removed_vif = []
    for _ in range(max_iter):
        if X_num.shape[1] <= 1:
            break  # nothing to compare against

        vif_df = _compute_vif(X_num)
        # Consider only drop candidates not in 'keep'
        candidates = vif_df[~vif_df['Feature'].isin(keep)]
        if candidates.empty:
            break

        worst = candidates.iloc[0]
        if not np.isfinite(worst['VIF']) or worst['VIF'] > thresh:
            col_to_drop = worst['Feature']
            X_num.drop(columns=[col_to_drop], inplace=True, errors='ignore')
            removed_vif.append(col_to_drop)
        else:
            # All non-kept features are under the threshold
            break

    # Build final dataset by dropping removed columns from the original X
    X_reduced = X.drop(columns=zero_var + removed_vif, errors='ignore')

    # Final VIF report on the remaining numeric columns
    final_vif_df = _compute_vif(X_reduced.select_dtypes(include=[np.number]))

    # Warn if any kept features are still above threshold
    high_kept = final_vif_df[(final_vif_df['Feature'].isin(keep)) & (final_vif_df['VIF'] > thresh)]
    if not high_kept.empty:
        print("⚠️ Some protected ('keep') features still exceed the VIF threshold:")
        print(high_kept)

    return X_reduced, zero_var, removed_vif, final_vif_df

# ==== Run on your data ====
# Ensure X_train1 is a DataFrame
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train)

# Optionally protect must-keep features, e.g. keep=["Risk_Score"]
X_train2, removed_zero_var, removed_by_vif, final_vif = reduce_vif(
    X_train,
    thresh=10.0,
    keep=[],          # e.g., ["Risk_Score"]
    max_iter=200
)

print(f"\nDropped zero-variance columns: {removed_zero_var}")
print(f"Dropped due to VIF>10: {removed_by_vif}")
print(f"X_train2 shape: {X_train2.shape}")
print("\nTop remaining VIFs:")
print(final_vif.head(20))

In [ ]:
y_train_final.shape,X_train.shape

In [ ]:
import statsmodels.api as sm_api

# Fit a logistic regression model to get p-values
logit_model = sm_api.Logit(y_train_final, sm_api.add_constant(X_train2)).fit(maxiter=100)

# Print the summary which includes p-values
print(logit_model.summary())

# Extract p-values from the model summary
p_values = logit_model.pvalues

# Display the p-values
print("\nP-values for each feature:\n", p_values)

# You can also sort and filter the p-values to display only significant features
significant_features = p_values[p_values < 0.05]
print("\nSignificant features (p < 0.05):\n", significant_features)

In [ ]:
params = logit_model.params
conf   = logit_model.conf_int()
pvals  = logit_model.pvalues

or_df = pd.DataFrame({
    "variable": params.index,
    "coef": params.values,
    "aOR": np.exp(params.values),
    "CI_low": np.exp(conf[0].values),
    "CI_high": np.exp(conf[1].values),
    "p_value": pvals.values
})

# drop intercept and keep significant terms
or_sig = (or_df
          .query("variable != 'const' and p_value < 0.05")
          .sort_values("p_value")
          .reset_index(drop=True))

# nice rounding for reporting
or_sig_rounded = or_sig.copy()
or_sig_rounded[["aOR","CI_low","CI_high"]] = or_sig_rounded[["aOR","CI_low","CI_high"]].round(3)
or_sig_rounded["coef"] = or_sig_rounded["coef"].round(3)
or_sig_rounded["p_value"] = or_sig_rounded["p_value"].apply(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")

or_sig_rounded


In [ ]:
sig = p_values.drop(['const', 'Intercept'], errors='ignore')
sig = sig[sig < 0.05].sort_values()
print(sig)
print("Names list:", sig.index.tolist())


In [ ]:
# significant_features is a Series: index = feature names, values = p-values
# Drop the intercept if present, and sort by p-value (optional)
sig_cols = (
    significant_features
      .drop(labels=['const'], errors='ignore')
      .sort_values()
      .index
      .tolist()
)

# Keep only columns that actually exist in X_train2
sig_cols = [c for c in sig_cols if c in X_train2.columns]

# Now subset safely
Xtrain = X_train2.loc[:, sig_cols]

print("Selected columns:", sig_cols)
print("Xtrain shape:", Xtrain.shape)


In [ ]:
Xtrain=X_train2[sig_cols]

In [ ]:
# Compute the correlation matrix
corr_matrix = Xtrain.corr()

# Create a mask to only show the lower triangle (or upper triangle)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Plot the heatmap
plt.figure(figsize=(30, 15))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='viridis', vmin=-1, vmax=1)
plt.show()


In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# Define model confusion matrices
models = {
    "CatBoost": {
        "TN": 38134,
        "FP": 8015,
        "FN": 383,
        "TP": 909
    },
    "LightGBM": {
        "TN": 37546,
        "FP": 8603,
        "FN": 319,
        "TP": 973
    },
    "XGBoost": {
        "TN": 14536,
        "FP": 31613,
        "FN": 476,
        "TP": 816
    },
    "Random Forest": {
        "TN": 39456,
        "FP": 6693,
        "FN": 360,
        "TP": 932
    }
}



# Expected validation performance
expected_performance = {
    "CatBoost": {
        "Accuracy": 0.823,
        "Recall": 0.704,
        "Precision": 0.102,
        "F1-score": 0.178,
        "AUC": 0.821
    },
    "LightGBM": {
        "Accuracy": 0.812,
        "Recall": 0.753,
        "Precision": 0.102,
        "F1-score": 0.179,
        "AUC": 0.835
    },
    "XGBoost": {
        "Accuracy": 0.324,
        "Recall": 0.632,
        "Precision": 0.025,
        "F1-score": 0.048,
        "AUC": 0.503
    },
    "Random Forest": {
        "Accuracy": 0.851,
        "Recall": 0.721,
        "Precision": 0.122,
        "F1-score": 0.209,
        "AUC": 0.839
    }
}




# Compute evaluation metrics
results = []
for model, cm in models.items():
    accuracy = (cm["TP"] + cm["TN"]) / sum(cm.values())
    recall = cm["TP"] / (cm["TP"] + cm["FN"])
    precision = cm["TP"] / (cm["TP"] + cm["FP"])
    f1_score = 2 * (precision * recall) / (precision + recall)
    specificity = cm["TN"] / (cm["TN"] + cm["FP"])
    
    auc = expected_performance[model]["AUC"]  # Use expected AUC values

    results.append([model, round(accuracy, 4), round(recall, 4), round(precision, 4), round(f1_score, 4), round(specificity, 4), round(auc, 4)])

# Convert to DataFrame
df1_results = pd.DataFrame(results, columns=["Model", "Accuracy", "Recall", "Precision", "F1-score", "Specificity", "AUC"])


df1_results
